# Analyze the AMG benchmark data

In [1]:
! pip install polars --quiet

In [2]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
MAIN_DIR = ROOT_DIR.joinpath("metabolism_benchmark")
AMG_TABLES_DIR = MAIN_DIR.joinpath("amg_tables")
OUTPUT_TABLES_DIR = Path("./tables/amg_benchmark")
OUTPUT_TABLES_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(10)

polars.config.Config

In [4]:
AMG_PREDICTIONS_PARQUET = AMG_TABLES_DIR.joinpath("amg_predictions_combined.parquet")

In [5]:
amg_predictions = pl.read_parquet(AMG_PREDICTIONS_PARQUET)
amg_predictions

gene,scaffold,source,sample,ecosystem,genomad_viral,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,in_strict_viral_region,CheckAMG_high,CheckAMG_medium,CheckAMG_low,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F,VIBRANT_high,VIBRANT_medium,VIBRANT_low
str,str,str,str,str,bool,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""Ga0485157_0000001_10""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,5259.0,3531.0,null,null,null,null,false,false,false,true,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_100""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,12252.0,6792.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_101""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,594.0,6198.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_102""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,816.0,5976.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""Ga0485157_0000001_103""","""Ga0485157_0000001""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,1161.0,5631.0,null,null,null,null,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_5""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,5847.0,4218.0,null,47403.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""scaffold_9_c1_6""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,6696.0,3369.0,null,46554.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false
"""scaffold_9_c1_7""","""scaffold_9_c1""","""Viromes""","""soil_T42_15_3_45""","""Soil""",false,7674.0,2391.0,null,45576.0,null,10.0,false,false,false,false,false,false,false,true,4,true,true,false,false,false,true,false,false,false,false


In [6]:
DEFAULT_TIERS = {
    "CheckAMG_high": "CheckAMG (high)",
    "DRAMV_default": "DRAM-V (default)",
    "VIBRANT_high":  "VIBRANT (high)",
}

ALL_TIERS = {
    "CheckAMG_high": "CheckAMG (high)",
    "CheckAMG_medium": "CheckAMG (medium)",
    "CheckAMG_low": "CheckAMG (low)",
    "DRAMV_default": "DRAM-V (default)",
    "DRAMV_allow_T": "DRAM-V (allow T)",
    "DRAMV_aux_4": "DRAM-V (aux <= 4)",
    "DRAMV_aux_4_allow_T": "DRAM-V (aux <= 4, allow T)",
    "VIBRANT_high":  "VIBRANT (high)",
    "VIBRANT_medium":  "VIBRANT (medium)",
    "VIBRANT_low":  "VIBRANT (low)",
}

SOURCE_LABELS = {
    "complete_virus_genomes": "Viral genomes",
    "metagenomes":            "Mixed metagenomes",
    "viromes":                "Viromes",
}

ECO_LABELS = {
    "gut":        "Human gut",
    "marine":     "Aquatic",
    "freshwater": "Aquatic",
    "soil":       "Soil",
}

## Get AMG counts

### Per tool x tier

Every tool has multiple tiers. DRAM-V's are stringency flags (default, allow T flag, aux score 4, aux 4 + T), not rank-ordered confidences like CheckAMG and VIBRANT. CheckAMG and VIBRANT tier as high / medium / low. Default / high are the advertised outputs.

In [7]:
count_rows = []
for tier in ALL_TIERS.keys():
    n = int(amg_predictions.select(pl.col(tier).sum()).item())
    n_viral = int(amg_predictions.filter(pl.col(tier) & pl.col("genomad_viral")).height)
    tool = tier.split("_")[0] if not tier.startswith("DRAMV") else "DRAM-V"
    count_rows.append({
        "tool": "CheckAMG" if tier.startswith("CheckAMG") else ("VIBRANT" if tier.startswith("VIBRANT") else "DRAM-V"),
        "tier": tier,
        "n_amg": n,
        "n_amg_genomad_viral": n_viral,
        "pct_genomad_viral": round(100.0 * n_viral / n, 3) if n else 0.0,
    })
counts_tier = pl.DataFrame(count_rows).sort(["tool", "tier"])

In [8]:
counts_tier

tool,tier,n_amg,n_amg_genomad_viral,pct_genomad_viral
str,str,i64,i64,f64
"""CheckAMG""","""CheckAMG_high""",3218,3151,97.918
"""CheckAMG""","""CheckAMG_low""",169511,8847,5.219
"""CheckAMG""","""CheckAMG_medium""",6706,5825,86.863
"""DRAM-V""","""DRAMV_allow_T""",10736,9846,91.71
"""DRAM-V""","""DRAMV_aux_4""",116657,61378,52.614
"""DRAM-V""","""DRAMV_aux_4_allow_T""",1900029,1199111,63.11
"""DRAM-V""","""DRAMV_default""",664,621,93.524
"""VIBRANT""","""VIBRANT_high""",1270,1263,99.449
"""VIBRANT""","""VIBRANT_low""",12641,12497,98.861


In [9]:
counts_tier.write_csv(OUTPUT_TABLES_DIR.joinpath("AMG_counts_by_tier.tsv"), separator="\t")

### Per source x ecosystem

In [10]:
srcs_eco = []
for tier in ALL_TIERS.keys():
    g = (
        amg_predictions
        .group_by(["source", "ecosystem"])
        .agg([
            pl.col(tier).sum().alias("n_amg"),
            (pl.col(tier) & pl.col("genomad_viral")).sum().alias("n_amg_genomad_viral"),
        ])
        .with_columns([
            pl.lit(tier).alias("tier"),
            pl.when(pl.col("n_amg") > 0)
            .then(100.0 * pl.col("n_amg_genomad_viral") / pl.col("n_amg"))
            .otherwise(0.0)
            .round(3)
            .alias("pct_genomad_viral"),
        ])
    )
    srcs_eco.append(g)

counts_source_eco = pl.concat(srcs_eco, how="vertical_relaxed")

counts_source_eco = counts_source_eco.sort(
    ["n_amg", "n_amg_genomad_viral", "source", "ecosystem", "tier"],
    descending=[True, True, False, False, False]
)

In [11]:
counts_source_eco

source,ecosystem,n_amg,n_amg_genomad_viral,tier,pct_genomad_viral
str,str,u64,u64,str,f64
"""Mixed metagenomes""","""Soil""",606564,596406,"""DRAMV_aux_4_allow_T""",98.325
"""Viromes""","""Aquatic""",491455,323098,"""DRAMV_aux_4_allow_T""",65.743
"""Viromes""","""Human gut""",293055,29180,"""DRAMV_aux_4_allow_T""",9.957
"""Mixed metagenomes""","""Aquatic""",254973,75108,"""DRAMV_aux_4_allow_T""",29.457
"""Viral genomes""","""Human gut""",63275,63272,"""DRAMV_aux_4_allow_T""",99.995
…,…,…,…,…,…
"""Mixed metagenomes""","""Human gut""",2,2,"""VIBRANT_high""",100.0
"""Viromes""","""Soil""",2,0,"""DRAMV_default""",0.0
"""Viral genomes""","""Aquatic""",1,1,"""DRAMV_default""",100.0


## Calculate overlapping/unique AMG calls

### Default/high-confidence tiers

Keep genes with at least one default/high call.

In [12]:
default_cols = list(DEFAULT_TIERS.keys())
any_default = pl.any_horizontal([pl.col(c) for c in default_cols])

members = (
    amg_predictions.filter(any_default)
        .select(
            "gene", "scaffold", "source", "sample", "ecosystem",
            "genomad_viral", "in_strict_viral_region",
            "viral_gene_left_dist", "viral_gene_right_dist",
            "MGE_gene_left_dist", "MGE_gene_right_dist",
            "MGE_gene_left_V_score", "MGE_gene_right_V_score",
            *ALL_TIERS.keys(),
            "DRAMV_auxiliary_score",
            "DRAMV_M", "DRAMV_T", "DRAMV_V", "DRAMV_A", "DRAMV_P", "DRAMV_B", "DRAMV_F",
        )
)

In [13]:
members

gene,scaffold,source,sample,ecosystem,genomad_viral,in_strict_viral_region,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,CheckAMG_high,CheckAMG_medium,CheckAMG_low,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,VIBRANT_high,VIBRANT_medium,VIBRANT_low,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F
str,str,str,str,str,bool,bool,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool
"""Ga0485157_0000020_8""","""Ga0485157_0000020""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,true,237.0,13092.0,null,null,null,null,true,true,true,false,false,false,false,false,false,false,null,false,false,false,false,false,false,false
"""Ga0485157_0000095_62""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,2910.0,1179.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_70""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,5556.0,2451.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_73""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,2451.0,3576.0,null,null,null,null,true,true,true,false,false,false,true,false,false,true,4,true,true,false,false,false,true,false
"""Ga0485157_0000261_37""","""Ga0485157_0000261""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",false,false,1698.0,1026.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_400_c1_20""","""scaffold_400_c1""","""Viromes""","""soil_T42_15_2_44""","""Soil""",true,true,2337.0,1020.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,true
"""scaffold_4830_c1_3""","""scaffold_4830_c1""","""Viromes""","""soil_T42_15_2_44""","""Soil""",false,true,252.0,2139.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,true
"""scaffold_806_c1_10""","""scaffold_806_c1""","""Viromes""","""soil_T42_15_2_44""","""Soil""",false,false,6708.0,null,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,true


In [14]:
members.write_csv(OUTPUT_TABLES_DIR.joinpath("AMG_default_high_members.tsv"), separator="\t")

In [15]:
members_viral = members.filter(pl.col("genomad_viral"))

In [16]:
members_viral

gene,scaffold,source,sample,ecosystem,genomad_viral,in_strict_viral_region,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,CheckAMG_high,CheckAMG_medium,CheckAMG_low,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,VIBRANT_high,VIBRANT_medium,VIBRANT_low,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F
str,str,str,str,str,bool,bool,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool
"""Ga0485157_0000095_62""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,2910.0,1179.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_70""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,5556.0,2451.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_73""","""Ga0485157_0000095""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,2451.0,3576.0,null,null,null,null,true,true,true,false,false,false,true,false,false,true,4,true,true,false,false,false,true,false
"""Ga0485157_0000331_45""","""Ga0485157_0000331""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,4770.0,1407.0,5847.0,null,10.0,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000578_2""","""Ga0485157_0000578""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""",true,true,null,498.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_37_c1_5""","""scaffold_37_c1""","""Viromes""","""soil_T42_15_1_43""","""Soil""",true,true,null,765.0,null,14646.0,null,10.0,false,false,false,false,false,false,false,true,true,true,null,false,false,false,false,false,false,false
"""scaffold_6199_c1_12""","""scaffold_6199_c1""","""Viromes""","""soil_T42_15_1_43""","""Soil""",true,false,null,null,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,true
"""scaffold_8_c1_7""","""scaffold_8_c1""","""Viromes""","""soil_T42_15_1_43""","""Soil""",true,true,4701.0,6342.0,null,40503.0,null,10.0,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false


In [17]:

from itertools import combinations

def set_counts(frame, tiers, name):

    rows = []
    cols = list(tiers.keys())

    for r in range(1, len(cols) + 1):
        for combo in combinations(cols, r):
            expr = pl.lit(True)

            for col in combo:
                expr = expr & pl.col(col)

            for col in cols:
                if col not in combo:
                    expr = expr & ~pl.col(col)

            rows.append({
                "set": " & ".join(tiers[c] for c in combo),
                "n_genes": int(frame.filter(expr).height),
                "scope": name,
            })

    union_expr = pl.lit(False)
    for col in cols:
        union_expr = union_expr | pl.col(col)

    rows.append({
        "set": "Total (any selected tier)",
        "n_genes": int(frame.filter(union_expr).height),
        "scope": name,
    })

    return pl.DataFrame(rows)

In [18]:
overall = set_counts(members, DEFAULT_TIERS, "all")

In [19]:
overall

set,n_genes,scope
str,i64,str
"""CheckAMG (high)""",2997,"""all"""
"""DRAM-V (default)""",664,"""all"""
"""VIBRANT (high)""",1049,"""all"""
"""CheckAMG (high) & DRAM-V (default)""",0,"""all"""
"""CheckAMG (high) & VIBRANT (high)""",221,"""all"""
"""DRAM-V (default) & VIBRANT (high)""",0,"""all"""
"""CheckAMG (high) & DRAM-V (default) & VIBRANT (high)""",0,"""all"""
"""Total (any selected tier)""",4931,"""all"""


In [20]:
viral = set_counts(members_viral, DEFAULT_TIERS, "genomad_viral")

In [21]:
viral

set,n_genes,scope
str,i64,str
"""CheckAMG (high)""",2930,"""genomad_viral"""
"""DRAM-V (default)""",621,"""genomad_viral"""
"""VIBRANT (high)""",1042,"""genomad_viral"""
"""CheckAMG (high) & DRAM-V (default)""",0,"""genomad_viral"""
"""CheckAMG (high) & VIBRANT (high)""",221,"""genomad_viral"""
"""DRAM-V (default) & VIBRANT (high)""",0,"""genomad_viral"""
"""CheckAMG (high) & DRAM-V (default) & VIBRANT (high)""",0,"""genomad_viral"""
"""Total (any selected tier)""",4814,"""genomad_viral"""


### All tiers

In [22]:
members_any = (
    amg_predictions
    .select(
        "gene", "scaffold", "source", "sample", "ecosystem",
        "genomad_viral", "in_strict_viral_region",
        "viral_gene_left_dist", "viral_gene_right_dist",
        "MGE_gene_left_dist", "MGE_gene_right_dist",
        "MGE_gene_left_V_score", "MGE_gene_right_V_score",
        *ALL_TIERS.keys(),
        "DRAMV_auxiliary_score",
        "DRAMV_M", "DRAMV_T", "DRAMV_V", "DRAMV_A", "DRAMV_P", "DRAMV_B", "DRAMV_F",
    )
)

In [23]:
members_any_viral = members_any.filter(pl.col("genomad_viral"))

In [24]:
overall_any = set_counts(members_any, ALL_TIERS, "all").sort(["n_genes"], descending=True)

In [25]:
overall_any

set,n_genes,scope
str,i64,str
"""Total (any selected tier)""",1977870,"""all"""
"""DRAM-V (aux <= 4, allow T)""",1679639,"""all"""
"""DRAM-V (aux <= 4) & DRAM-V (aux <= 4, allow T)""",108149,"""all"""
"""CheckAMG (low) & DRAM-V (aux <= 4, allow T)""",79664,"""all"""
"""CheckAMG (low)""",75423,"""all"""
…,…,…
"""CheckAMG (high) & CheckAMG (medium) & CheckAMG (low) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM-V…",0,"""all"""
"""CheckAMG (high) & CheckAMG (medium) & DRAM-V (default) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM…",0,"""all"""
"""CheckAMG (high) & CheckAMG (low) & DRAM-V (default) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM-V …",0,"""all"""


In [26]:
viral_any = set_counts(members_viral, ALL_TIERS, "genomad_viral").sort(["n_genes"], descending=True)

In [27]:
viral_any

set,n_genes,scope
str,i64,str
"""Total (any selected tier)""",4814,"""genomad_viral"""
"""CheckAMG (high) & CheckAMG (medium) & CheckAMG (low) & DRAM-V (aux <= 4, allow T)""",1663,"""genomad_viral"""
"""DRAM-V (aux <= 4, allow T) & VIBRANT (high) & VIBRANT (medium) & VIBRANT (low)""",884,"""genomad_viral"""
"""CheckAMG (high) & CheckAMG (medium) & CheckAMG (low) & DRAM-V (aux <= 4, allow T) & VIBRANT (low)""",644,"""genomad_viral"""
"""DRAM-V (default) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM-V (aux <= 4, allow T)""",615,"""genomad_viral"""
…,…,…
"""CheckAMG (high) & CheckAMG (medium) & CheckAMG (low) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM-V…",0,"""genomad_viral"""
"""CheckAMG (high) & CheckAMG (medium) & DRAM-V (default) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM…",0,"""genomad_viral"""
"""CheckAMG (high) & CheckAMG (low) & DRAM-V (default) & DRAM-V (allow T) & DRAM-V (aux <= 4) & DRAM-V …",0,"""genomad_viral"""


### Pairwise jaccard and overlap percentages

In [28]:
def pairwise(frame):
    rows = []
    sets = {
        "CheckAMG (high)": pl.col("CheckAMG_high"),
        "CheckAMG (medium)": pl.col("CheckAMG_medium"),
        "CheckAMG (low)": pl.col("CheckAMG_low"),
        "DRAM-V (default)": pl.col("DRAMV_default"),
        "DRAM-V (allow T)": pl.col("DRAMV_allow_T"),
        "DRAM-V (aux <= 4)": pl.col("DRAMV_aux_4"),
        "DRAM-V (aux <= 4, allow T)": pl.col("DRAMV_aux_4_allow_T"),
        "VIBRANT (high)":  pl.col("VIBRANT_high"),
        "VIBRANT (medium)":  pl.col("VIBRANT_medium"),
        "VIBRANT (low)":  pl.col("VIBRANT_low"),
    }
    for a_name, a_expr in sets.items():
        for b_name, b_expr in sets.items():
            if a_name >= b_name:
                continue
            na = frame.filter(a_expr).height
            nb = frame.filter(b_expr).height
            nab = frame.filter(a_expr & b_expr).height
            union = frame.filter(a_expr | b_expr).height
            rows.append({
                "tier_a": a_name, "tier_b": b_name,
                "n_a": na, "n_b": nb, "n_intersection": nab, "n_union": union,
                "jaccard": round(nab / union, 4) if union else 0.0,
                "pct_a_in_b": round(100.0 * nab / na, 2) if na else 0.0,
                "pct_b_in_a": round(100.0 * nab / nb, 2) if nb else 0.0,
            })
    return pl.DataFrame(rows)

In [29]:
pw = pairwise(members_any)
pw

tier_a,tier_b,n_a,n_b,n_intersection,n_union,jaccard,pct_a_in_b,pct_b_in_a
str,str,i64,i64,i64,i64,f64,f64,f64
"""CheckAMG (high)""","""CheckAMG (medium)""",3218,6706,3218,6706,0.4799,100.0,47.99
"""CheckAMG (high)""","""CheckAMG (low)""",3218,169511,3218,169511,0.019,100.0,1.9
"""CheckAMG (high)""","""DRAM-V (default)""",3218,664,0,3882,0.0,0.0,0.0
"""CheckAMG (high)""","""DRAM-V (allow T)""",3218,10736,21,13933,0.0015,0.65,0.2
"""CheckAMG (high)""","""DRAM-V (aux <= 4)""",3218,116657,118,119757,0.001,3.67,0.1
…,…,…,…,…,…,…,…,…
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (medium)""",1900029,3720,3374,1900375,0.0018,0.18,90.7
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (low)""",1900029,12641,10842,1901828,0.0057,0.57,85.77
"""VIBRANT (high)""","""VIBRANT (medium)""",1270,3720,1270,3720,0.3414,100.0,34.14


In [30]:
pw_v = pairwise(members_any_viral)
pw_v

tier_a,tier_b,n_a,n_b,n_intersection,n_union,jaccard,pct_a_in_b,pct_b_in_a
str,str,i64,i64,i64,i64,f64,f64,f64
"""CheckAMG (high)""","""CheckAMG (medium)""",3151,5825,3151,5825,0.5409,100.0,54.09
"""CheckAMG (high)""","""CheckAMG (low)""",3151,8847,3151,8847,0.3562,100.0,35.62
"""CheckAMG (high)""","""DRAM-V (default)""",3151,621,0,3772,0.0,0.0,0.0
"""CheckAMG (high)""","""DRAM-V (allow T)""",3151,9846,21,12976,0.0016,0.67,0.21
"""CheckAMG (high)""","""DRAM-V (aux <= 4)""",3151,61378,111,64418,0.0017,3.52,0.18
…,…,…,…,…,…,…,…,…
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (medium)""",1199111,3708,3362,1199457,0.0028,0.28,90.67
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (low)""",1199111,12497,10740,1200868,0.0089,0.9,85.94
"""VIBRANT (high)""","""VIBRANT (medium)""",1263,3708,1263,3708,0.3406,100.0,34.06


In [31]:
pairwise_df = pl.concat(
    [pw.with_columns(pl.lit("all").alias("scope")), pw_v.with_columns(pl.lit("genomad_viral").alias("scope"))],
    how="vertical_relaxed"
)
pairwise_df

tier_a,tier_b,n_a,n_b,n_intersection,n_union,jaccard,pct_a_in_b,pct_b_in_a,scope
str,str,i64,i64,i64,i64,f64,f64,f64,str
"""CheckAMG (high)""","""CheckAMG (medium)""",3218,6706,3218,6706,0.4799,100.0,47.99,"""all"""
"""CheckAMG (high)""","""CheckAMG (low)""",3218,169511,3218,169511,0.019,100.0,1.9,"""all"""
"""CheckAMG (high)""","""DRAM-V (default)""",3218,664,0,3882,0.0,0.0,0.0,"""all"""
"""CheckAMG (high)""","""DRAM-V (allow T)""",3218,10736,21,13933,0.0015,0.65,0.2,"""all"""
"""CheckAMG (high)""","""DRAM-V (aux <= 4)""",3218,116657,118,119757,0.001,3.67,0.1,"""all"""
…,…,…,…,…,…,…,…,…,…
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (medium)""",1199111,3708,3362,1199457,0.0028,0.28,90.67,"""genomad_viral"""
"""DRAM-V (aux <= 4, allow T)""","""VIBRANT (low)""",1199111,12497,10740,1200868,0.0089,0.9,85.94,"""genomad_viral"""
"""VIBRANT (high)""","""VIBRANT (medium)""",1263,3708,1263,3708,0.3406,100.0,34.06,"""genomad_viral"""


In [32]:
(
    pairwise_df
    .filter(
        (pl.col("tier_a").is_in(DEFAULT_TIERS.values())) & (pl.col("tier_b").is_in(DEFAULT_TIERS.values()))
    )
    .sort(["scope", "jaccard"], descending=[False, True])
)

tier_a,tier_b,n_a,n_b,n_intersection,n_union,jaccard,pct_a_in_b,pct_b_in_a,scope
str,str,i64,i64,i64,i64,f64,f64,f64,str
"""CheckAMG (high)""","""VIBRANT (high)""",3218,1270,221,4267,0.0518,6.87,17.4,"""all"""
"""CheckAMG (high)""","""DRAM-V (default)""",3218,664,0,3882,0.0,0.0,0.0,"""all"""
"""DRAM-V (default)""","""VIBRANT (high)""",664,1270,0,1934,0.0,0.0,0.0,"""all"""
"""CheckAMG (high)""","""VIBRANT (high)""",3151,1263,221,4193,0.0527,7.01,17.5,"""genomad_viral"""
"""CheckAMG (high)""","""DRAM-V (default)""",3151,621,0,3772,0.0,0.0,0.0,"""genomad_viral"""
"""DRAM-V (default)""","""VIBRANT (high)""",621,1263,0,1884,0.0,0.0,0.0,"""genomad_viral"""


## Load and format gene annotations and get per-gene final annotations per tool

Aggregate per-gene lists of hit ids and databases across each tool's `final_annot` annotations. CheckAMG and VIBRANT keep one best hit per gene x reference database, while DRAM-V accepts every hit that passes its filters so multi-domain proteins contribute multiple rows per gene (mostly in Pfam). We keep the multiplicity and test set membership downstream rather than picking a single best hit.

In [33]:
annotations = pl.read_parquet(AMG_TABLES_DIR.joinpath("amg_all_annotations.parquet"))

In [34]:
annotations

gene,source,sample,ecosystem,tool,database,hit_id,hit_desc,bitscore,evalue,final_annot
str,str,str,str,str,str,str,str,f64,f64,bool
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""KEGG""","""K04708""","""KDSR, kdsr; 3-dehydrosphinganine reductase [EC:1.1.1.102]""",297.572968,2.4026e-87,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""KEGG""","""K07124""","""K07124; uncharacterized protein""",185.187241,3.0093e-53,false
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""CAMPER""","""K07535""","""2-hydroxycyclohexanecarboxyl-CoA dehydrogenase [EC:1.1.1.-]""",178.950897,2.1448e-51,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""CAMPER""","""K07535""","""badH; 2-hydroxycyclohexanecarboxyl-CoA dehydrogenase [EC:1.1.1.-]""",178.950897,2.1448e-51,true
"""Ga0485157_0000001_10""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""Pfam""","""PF00106""","""short chain dehydrogenase""",177.693161,2.3708e-51,true
…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K02071""","""metN; D-methionine transport system ATP-binding protein""",41.1,2.1000e-11,false
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K10558""","""lsrA, ego; AI-2 transport system ATP-binding protein""",40.3,2.5000e-11,false
"""scaffold_9_c1_8""","""Viromes""","""soil_T42_15_3_45""","""Soil""","""VIBRANT""","""KEGG""","""K10021""","""occP, nocP; octopine/nopaline transport system ATP-binding protein [EC:7.4.2.1]""",40.4,2.8000e-11,false


In [35]:
annotations_high_default = (
    members
    .join(
        annotations,
        left_on=["source", "sample", "ecosystem", "gene"],
        right_on=["source", "sample", "ecosystem", "gene"],
        how = "left"
    )
)

In [36]:
DB_SUPPORT = {
    "CheckAMG": {"CAMPER", "FOAM", "KEGG", "METABOLIC", "PHROG", "Pfam", "dbCAN"},
    "DRAMV":    {"KEGG", "Pfam", "dbCAN", "VOG", "MEROPS", "RefSeq Viral"},
    "VIBRANT":  {"KEGG", "Pfam", "VOG"},
}

ANNOT_TOOL = {"CheckAMG": "CheckAMG", "DRAMV": "DRAM-V", "VIBRANT": "VIBRANT"}

In [37]:
def likely_false_positive_expr():
    """Gene lacks viral evidence: no genomad viral, no strict region, no nearby viral gene."""
    has_left_viral = pl.col("viral_gene_left_dist").is_not_null()
    has_right_viral = pl.col("viral_gene_right_dist").is_not_null()
    return ~(pl.col("genomad_viral") | pl.col("in_strict_viral_region") | has_left_viral | has_right_viral)

def hits_per_gene(subset: pl.DataFrame, tool: str, prefix: str | None = None) -> pl.DataFrame:
    """Per-gene set of hit_ids and databases for a given annotation-tool string.
    Column names use `prefix` (defaults to `tool`) so tier-prefix-style names
    like DRAMV_hit_ids line up with tier columns.
    """
    pref = prefix or tool
    return (
        subset.filter(pl.col("tool") == tool)
              .group_by("gene", "source", "sample", "ecosystem")
              .agg([
                  pl.col("hit_id").unique().alias(f"{pref}_hit_ids"),
                  pl.col("database").unique().alias(f"{pref}_hit_dbs"),
                  pl.len().alias(f"{pref}_n_hits"),
              ])
    )

def final_hits_per_gene(df: pl.DataFrame, tool: str, prefix: str) -> pl.DataFrame:
    """Per-gene sets of (final_annot=True) hit_ids and databases for a tool."""
    return (
        df.filter((pl.col("tool") == tool) & pl.col("final_annot"))
             .group_by("gene", "source", "sample", "ecosystem")
             .agg([
                 pl.col("hit_id").unique().alias(f"{prefix}_final_hit_ids"),
                 pl.col("database").unique().alias(f"{prefix}_final_hit_dbs"),
                 pl.len().alias(f"{prefix}_n_final_hits"),
             ])
    )

In [38]:
checkamg_hits = hits_per_gene(annotations_high_default, "CheckAMG")
dramv_hits = hits_per_gene(annotations_high_default, ANNOT_TOOL["DRAMV"], "DRAMV")
vibrant_hits = hits_per_gene(annotations_high_default, "VIBRANT")

In [39]:
checkamg_final = final_hits_per_gene(annotations_high_default, ANNOT_TOOL["CheckAMG"], "CheckAMG")
dramv_final = final_hits_per_gene(annotations_high_default, ANNOT_TOOL["DRAMV"], "DRAMV")
vibrant_final = final_hits_per_gene(annotations_high_default, ANNOT_TOOL["VIBRANT"], "VIBRANT")

In [40]:
checkamg_final

gene,source,sample,ecosystem,CheckAMG_final_hit_ids,CheckAMG_final_hit_dbs,CheckAMG_n_final_hits
str,str,str,str,list[str],list[str],u64
"""IMGVR_UViG_3300045988_166091_3300045988_Ga0495776_005041_2""","""Viral genomes""","""virus_genomes_gut""","""Human gut""","[""PF08645"", ""HMMsoil57411""]","[""FOAM"", ""Pfam""]",2
"""IMGVR_UViG_3300045988_085863_3300045988_Ga0495776_142020_49""","""Viral genomes""","""virus_genomes_gut""","""Human gut""","[""PF01507"", ""HMMsoil57593"", … ""phrog_424""]","[""PHROG"", ""KEGG"", … ""Pfam""]",5
"""scaffold_55_c1_63""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","[""phrog_363"", ""K00558"", ""PF00145""]","[""KEGG"", ""PHROG"", ""Pfam""]",4
"""scaffold_1015_c1_44""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""phrog_2197"", ""PF01227"", ""K01495""]","[""KEGG"", ""Pfam"", ""PHROG""]",3
"""IMGVR_UViG_3300048581_000026_3300048581_Ga0499404_0506_16""","""Viral genomes""","""virus_genomes_soil""","""Soil""","[""PF13561""]","[""Pfam""]",1
…,…,…,…,…,…,…
"""Ga0485175_0000410_24""","""Viromes""","""freshwater_Ga0485175_contigs""","""Aquatic""","[""PF02543"", ""K00612"", ""phrog_1601""]","[""PHROG"", ""KEGG"", ""Pfam""]",3
"""scaffold_858_c1_24""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","[""PF03102""]","[""Pfam""]",1
"""Ga0485173_0000029_28""","""Viromes""","""freshwater_Ga0485173_contigs""","""Aquatic""","[""PF02666"", ""phrog_12670""]","[""Pfam"", ""PHROG""]",2


In [41]:
dramv_final

gene,source,sample,ecosystem,DRAMV_final_hit_ids,DRAMV_final_hit_dbs,DRAMV_n_final_hits
str,str,str,str,list[str],list[str],u64
"""Ga0485175_0001214_31""","""Viromes""","""freshwater_Ga0485175_contigs""","""Aquatic""","[""PF00109"", ""PF00676"", … ""PF00490""]","[""Pfam""]",6
"""scaffold_3582_c1_5""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","[""PF03435"", ""PF01370"", … ""PF02675""]","[""Pfam""]",10
"""scaffold_2912_c1_15""","""Mixed metagenomes""","""soil_T42_15_1_55""","""Soil""","[""PF02445"", ""PF02350"", … ""PF01227""]","[""Pfam""]",4
"""scaffold_896_c1_20""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""PF03659"", ""PF03171"", … ""PF01384""]","[""Pfam""]",4
"""Ga0485173_0000271_47""","""Viromes""","""freshwater_Ga0485173_contigs""","""Aquatic""","[""PF11775"", ""PF13640"", … ""PF01050""]","[""Pfam""]",7
…,…,…,…,…,…,…
"""scaffold_703_c1_26""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""PF02274"", ""PF04896"", … ""PF10396""]","[""Pfam""]",10
"""Ga0485174_0008544_6""","""Viromes""","""freshwater_Ga0485174_contigs""","""Aquatic""","[""PF00120"", ""PF03951"", … ""PF01120""]","[""Pfam""]",6
"""IMGVR_UViG_3300045988_116494_3300045988_Ga0495776_122541_102""","""Viral genomes""","""virus_genomes_gut""","""Human gut""","[""PF01761"", ""PF01663"", … ""PF04095""]","[""Pfam""]",4


In [42]:
vibrant_final

gene,source,sample,ecosystem,VIBRANT_final_hit_ids,VIBRANT_final_hit_dbs,VIBRANT_n_final_hits
str,str,str,str,list[str],list[str],u64
"""Ga0485173_0000008_166""","""Viromes""","""freshwater_Ga0485173_contigs""","""Aquatic""","[""K12452"", ""PF01041""]","[""Pfam"", ""KEGG""]",2
"""scaffold_1410_c1_46""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""K06720"", ""PF06339""]","[""Pfam"", ""KEGG""]",2
"""scaffold_514_c1_37""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","[""K00059"", ""PF00106""]","[""Pfam"", ""KEGG""]",2
"""scaffold_304_c1_11""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""PF03102"", ""K01654""]","[""KEGG"", ""Pfam""]",2
"""IMGVR_UViG_3300012862_000037_3300012862_Ga0160483_1000225_5420-48970_13""","""Viral genomes""","""virus_genomes_soil""","""Soil""","[""K00558"", ""PF00145""]","[""KEGG"", ""Pfam""]",2
…,…,…,…,…,…,…
"""scaffold_216_c1_43""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""PF03275"", ""K01854""]","[""Pfam"", ""KEGG""]",2
"""scaffold_1118_c1_17""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","[""PF01227"", ""K01495""]","[""KEGG"", ""Pfam""]",2
"""scaffold_398_c1_38""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","[""PF01227"", ""K01495""]","[""KEGG"", ""Pfam""]",2


## Why do other tools not call the same gene AMG?

For every gene called AMG at default / high by one tool, classify why each other tool did not agree. Each source tier produces two classifications (one per other tool), based on six mutually exclusive categories plus a "lacks viral evidence" flag and an "agrees at default / high" pass-through:

- **a.** Called at a different tier
- **b.** Lacks viral evidence
- **c.** Source tool's best-hit database not queried by the other tool
- **d.** Shared-database hit id between the two tools, but the other tool did not flag it as an AMG
- **e.** Other tool has a hit for the gene in a shared database but returned a different top hit
- **f.** Other tool produced no annotation at all

The source tool's set of `final_annot` hit ids is compared against the other tool's entire raw annotation set, so a tool that hit the gene in any shared database is credited. Database-overlap checks use the union of the source tool's final databases. Category `c` applies only when none of those databases is queried by the other tool.

In [43]:
SOURCE_TIERS = [
    ("CheckAMG_high", "CheckAMG"),
    ("DRAMV_default", "DRAMV"),
    ("VIBRANT_high",  "VIBRANT"),
]

OTHER_TIER_FLAGS = {
    "CheckAMG": [pl.col("CheckAMG_medium"), pl.col("CheckAMG_low")],
    "DRAMV":    [pl.col("DRAMV_allow_T"), pl.col("DRAMV_aux_4"), pl.col("DRAMV_aux_4_allow_T")],
    "VIBRANT":  [pl.col("VIBRANT_medium"), pl.col("VIBRANT_low")],
}

AGREE_FLAG = {
    "CheckAMG": pl.col("CheckAMG_high"),
    "DRAMV":    pl.col("DRAMV_default"),
    "VIBRANT":  pl.col("VIBRANT_high"),
}

In [44]:
members_annot = (
    members
    .join(checkamg_hits,  on=["gene", "sample", "source", "ecosystem"], how="left")
    .join(dramv_hits,     on=["gene", "sample", "source", "ecosystem"], how="left")
    .join(vibrant_hits,   on=["gene", "sample", "source", "ecosystem"], how="left")
    .join(checkamg_final, on=["gene", "sample", "source", "ecosystem"], how="left")
    .join(dramv_final,    on=["gene", "sample", "source", "ecosystem"], how="left")
    .join(vibrant_final,  on=["gene", "sample", "source", "ecosystem"], how="left")
    .with_columns([
        pl.col(c).fill_null(0).cast(pl.Int64)
        for c in [
            "CheckAMG_n_hits", "DRAMV_n_hits", "VIBRANT_n_hits",
            "CheckAMG_n_final_hits", "DRAMV_n_final_hits", "VIBRANT_n_final_hits",
        ]
    ])
    .with_columns(likely_false_positive_expr().alias("likely_false_positive"))
)

In [45]:
def classify_nonoverlap(m: pl.DataFrame, source_tiers=SOURCE_TIERS,
                        return_per_gene: bool = False) -> pl.DataFrame:
    rows = []
    per_gene = []
    for src_tier, src_tool in source_tiers:
        src_set = m.filter(pl.col(src_tier))
        n_total = src_set.height
        n_b = int(src_set.filter(pl.col("likely_false_positive")).height)

        for other_tool in ["CheckAMG", "DRAMV", "VIBRANT"]:
            if other_tool == src_tool:
                continue

            agree_default_high = AGREE_FLAG[other_tool]
            other_any_tier = pl.any_horizontal([AGREE_FLAG[other_tool]] + OTHER_TIER_FLAGS[other_tool])

            src_hit_ids = pl.col(f"{src_tool}_final_hit_ids")
            src_dbs     = pl.col(f"{src_tool}_final_hit_dbs")
            other_hits  = pl.col(f"{other_tool}_hit_ids")
            other_n     = pl.col(f"{other_tool}_n_hits")

            shared_hit = (
                pl.when(src_hit_ids.is_null() | other_hits.is_null())
                  .then(False)
                  .otherwise(src_hit_ids.list.set_intersection(other_hits).list.len() > 0)
            )
            src_db_supported = (
                pl.when(src_dbs.is_null())
                  .then(False)
                  .otherwise(src_dbs.list.eval(pl.element().is_in(list(DB_SUPPORT[other_tool]))).list.any())
            )
            has_any_hits = other_n > 0

            cat_expr = (
                pl.when(other_any_tier & ~agree_default_high)
                  .then(pl.lit("a. other tool called at different tier"))
                  .when(agree_default_high)
                  .then(pl.lit("agrees at default/high"))
                  .when(shared_hit)
                  .then(pl.lit("d. same db hit_id, not flagged AMG"))
                  .when(has_any_hits & ~src_db_supported)
                  .then(pl.lit("c. source tool's best-hit db not used by other tool"))
                  .when(has_any_hits)
                  .then(pl.lit("e. other tool annotated different hit in shared db"))
                  .otherwise(pl.lit("f. other tool produced no hits for this gene"))
            )
            classified = src_set.with_columns(cat_expr.alias("category"))
            per_gene.append(classified.select([
                "gene", "source", "sample", "ecosystem",
                pl.lit(src_tier).alias("source_tier"),
                pl.lit(other_tool).alias("other_tool"),
                "category", "likely_false_positive",
            ]))
            counts = (
                classified.group_by("category").agg(pl.len().alias("n"))
                          .with_columns([
                              pl.lit(src_tier).alias("source_tier"),
                              pl.lit(other_tool).alias("other_tool"),
                              pl.lit(n_total).alias("source_total"),
                          ])
            )
            rows.append(counts)

        rows.append(pl.DataFrame({
            "category": ["b. gene lacks any viral evidence"],
            "n": [n_b],
            "source_tier": [src_tier],
            "other_tool": [None],
            "source_total": [n_total],
        }).cast({"n": pl.UInt32}))

    if return_per_gene:
        return pl.concat(per_gene, how="vertical_relaxed")
    return (
        pl.concat(rows, how="diagonal_relaxed")
          .with_columns((pl.col("n") / pl.col("source_total") * 100).round(2).alias("pct"))
    )

In [46]:
overlap_classification = (
    classify_nonoverlap(members_annot)
    .sort(["source_tier", "other_tool", "n"], descending=[False, False, True])
)

In [47]:
with pl.Config(tbl_rows=100, tbl_width_chars=800):
    print(overlap_classification)

shape: (28, 6)
┌─────────────────────────────────────────────────────┬──────┬───────────────┬────────────┬──────────────┬───────┐
│ category                                            ┆ n    ┆ source_tier   ┆ other_tool ┆ source_total ┆ pct   │
│ ---                                                 ┆ ---  ┆ ---           ┆ ---        ┆ ---          ┆ ---   │
│ str                                                 ┆ u64  ┆ str           ┆ str        ┆ i64          ┆ f64   │
╞═════════════════════════════════════════════════════╪══════╪═══════════════╪════════════╪══════════════╪═══════╡
│ b. gene lacks any viral evidence                    ┆ 0    ┆ CheckAMG_high ┆ null       ┆ 3218         ┆ 0.0   │
│ a. other tool called at different tier              ┆ 2988 ┆ CheckAMG_high ┆ DRAMV      ┆ 3218         ┆ 92.85 │
│ d. same db hit_id, not flagged AMG                  ┆ 188  ┆ CheckAMG_high ┆ DRAMV      ┆ 3218         ┆ 5.84  │
│ e. other tool annotated different hit in shared db  ┆ 28   ┆ Ch

## Which tier has more viral context when tools disagree?

For every gene called at default or high by one tool and at a different tier by another, compare genomic context: geNomad viral prediction, CheckAMG's strict viral region flag, flanking viral genes within 10 kb, and flanking MGE V-score (virus-like at V >= 10, host-like below).

In [48]:
VIRAL_VSCORE_MIN = 10
MAX_DIST = 10_000

In [49]:
def context_flags():
    near_viral_any = (
        (pl.col("viral_gene_left_dist").is_not_null() & (pl.col("viral_gene_left_dist") <= MAX_DIST))
        | (pl.col("viral_gene_right_dist").is_not_null() & (pl.col("viral_gene_right_dist") <= MAX_DIST))
    )
    near_viral_both = (
        pl.col("viral_gene_left_dist").is_not_null() & (pl.col("viral_gene_left_dist") <= MAX_DIST)
        & pl.col("viral_gene_right_dist").is_not_null() & (pl.col("viral_gene_right_dist") <= MAX_DIST)
    )
    near_mge_any = (
        (pl.col("MGE_gene_left_dist").is_not_null() & (pl.col("MGE_gene_left_dist") <= MAX_DIST))
        | (pl.col("MGE_gene_right_dist").is_not_null() & (pl.col("MGE_gene_right_dist") <= MAX_DIST))
    )
    # NaN V-scores are nulled first so an undefined V-score is not counted as virus-like
    _vl = pl.col("MGE_gene_left_V_score").fill_nan(None)
    _vr = pl.col("MGE_gene_right_V_score").fill_nan(None)
    near_virus_like_mge = (
        (_vl.is_not_null() & (_vl >= VIRAL_VSCORE_MIN))
        | (_vr.is_not_null() & (_vr >= VIRAL_VSCORE_MIN))
    )
    near_bact_like_mge = (
        (_vl.is_not_null() & (_vl < VIRAL_VSCORE_MIN))
        | (_vr.is_not_null() & (_vr < VIRAL_VSCORE_MIN))
    )
    return [
        near_viral_any.alias("near_viral_gene"),
        near_viral_both.alias("near_viral_gene_both_sides"),
        near_mge_any.alias("near_mge_gene"),
        near_virus_like_mge.alias("near_virus_like_mge"),
        near_bact_like_mge.alias("near_bact_like_mge"),
    ]

In [50]:
CONFLICT_SCENARIOS = [
    ("CheckAMG_high_vs_DRAMV_allow_T",
        pl.col("CheckAMG_high") & pl.col("DRAMV_allow_T") & ~pl.col("DRAMV_default")),
    ("CheckAMG_high_vs_DRAMV_aux_4",
        pl.col("CheckAMG_high") & pl.col("DRAMV_aux_4") & ~pl.col("DRAMV_default")),
    ("CheckAMG_high_vs_DRAMV_aux_4_allow_T",
        pl.col("CheckAMG_high") & pl.col("DRAMV_aux_4_allow_T")
        & ~pl.col("DRAMV_default") & ~pl.col("DRAMV_allow_T") & ~pl.col("DRAMV_aux_4")),
    ("VIBRANT_high_vs_DRAMV_allow_T",
        pl.col("VIBRANT_high") & pl.col("DRAMV_allow_T") & ~pl.col("DRAMV_default")),
    ("VIBRANT_high_vs_DRAMV_aux_4_allow_T",
        pl.col("VIBRANT_high") & pl.col("DRAMV_aux_4_allow_T")
        & ~pl.col("DRAMV_default") & ~pl.col("DRAMV_allow_T") & ~pl.col("DRAMV_aux_4")),
    ("DRAMV_default_vs_CheckAMG_low",
        pl.col("DRAMV_default") & pl.col("CheckAMG_low")),
    ("DRAMV_default_vs_VIBRANT_low",
        pl.col("DRAMV_default") & pl.col("VIBRANT_low")),
    ("CheckAMG_high_vs_VIBRANT_low",
        pl.col("CheckAMG_high") & pl.col("VIBRANT_low") & ~pl.col("VIBRANT_high")),
    ("VIBRANT_high_vs_CheckAMG_low",
        pl.col("VIBRANT_high") & pl.col("CheckAMG_low") & ~pl.col("CheckAMG_high")),
]

In [51]:
members_ctx = members_annot.with_columns(context_flags())

rows = []
for label, expr in CONFLICT_SCENARIOS:
    sub = members_ctx.filter(expr)
    n = sub.height
    if n == 0:
        rows.append({"scenario": label, "n_genes": 0})
        continue
    def ct(e):
        return int(sub.filter(e).height)
    rows.append({
        "scenario": label,
        "n_genes": n,
        "n_genomad_viral":         ct(pl.col("genomad_viral")),
        "n_strict_viral_region":   ct(pl.col("in_strict_viral_region")),
        "n_near_viral_gene":       ct(pl.col("near_viral_gene")),
        "n_near_viral_both_sides": ct(pl.col("near_viral_gene_both_sides")),
        "n_near_virus_like_mge":   ct(pl.col("near_virus_like_mge")),
        "n_near_bact_like_mge":    ct(pl.col("near_bact_like_mge")),
        "n_no_viral_evidence":     ct(~(pl.col("genomad_viral") | pl.col("in_strict_viral_region") | pl.col("near_viral_gene"))),
    })

amg_tier_conflict_df = pl.DataFrame(rows).with_columns([
    (pl.col("n_genomad_viral")         / pl.col("n_genes") * 100).round(1).alias("pct_genomad_viral"),
    (pl.col("n_strict_viral_region")   / pl.col("n_genes") * 100).round(1).alias("pct_strict_viral_region"),
    (pl.col("n_near_viral_gene")       / pl.col("n_genes") * 100).round(1).alias("pct_near_viral_gene"),
    (pl.col("n_near_viral_both_sides") / pl.col("n_genes") * 100).round(1).alias("pct_near_viral_both_sides"),
    (pl.col("n_near_virus_like_mge")   / pl.col("n_genes") * 100).round(1).alias("pct_near_virus_like_mge"),
    (pl.col("n_near_bact_like_mge")    / pl.col("n_genes") * 100).round(1).alias("pct_near_bact_like_mge"),
    (pl.col("n_no_viral_evidence")     / pl.col("n_genes") * 100).round(1).alias("pct_no_viral_evidence"),
])

In [52]:
amg_tier_conflict_df

scenario,n_genes,n_genomad_viral,n_strict_viral_region,n_near_viral_gene,n_near_viral_both_sides,n_near_virus_like_mge,n_near_bact_like_mge,n_no_viral_evidence,pct_genomad_viral,pct_strict_viral_region,pct_near_viral_gene,pct_near_viral_both_sides,pct_near_virus_like_mge,pct_near_bact_like_mge,pct_no_viral_evidence
str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""CheckAMG_high_vs_DRAMV_allow_T""",21,21,19,19,16,5,0,0,100.0,90.5,90.5,76.2,23.8,0.0,0.0
"""CheckAMG_high_vs_DRAMV_aux_4""",118,111,89,113,72,6,0,0,94.1,75.4,95.8,61.0,5.1,0.0,0.0
"""CheckAMG_high_vs_DRAMV_aux_4_allow_T""",2849,2800,2610,2815,2218,1006,30,0,98.3,91.6,98.8,77.9,35.3,1.1,0.0
"""VIBRANT_high_vs_DRAMV_allow_T""",8,8,8,8,7,5,0,0,100.0,100.0,100.0,87.5,62.5,0.0,0.0
"""VIBRANT_high_vs_DRAMV_aux_4_allow_T""",1135,1128,1086,1132,1015,781,15,0,99.4,95.7,99.7,89.4,68.8,1.3,0.0
"""DRAMV_default_vs_CheckAMG_low""",5,3,2,5,2,0,0,0,60.0,40.0,100.0,40.0,0.0,0.0,0.0
"""DRAMV_default_vs_VIBRANT_low""",5,5,4,5,3,0,0,0,100.0,80.0,100.0,60.0,0.0,0.0,0.0
"""CheckAMG_high_vs_VIBRANT_low""",1053,1050,958,1043,782,313,10,0,99.7,91.0,99.1,74.3,29.7,0.9,0.0
"""VIBRANT_high_vs_CheckAMG_low""",54,51,30,54,30,30,0,0,94.4,55.6,100.0,55.6,55.6,0.0,0.0


In [53]:
rows = []
for label in ALL_TIERS:
    sub = members_ctx.filter(pl.col(label))
    tool = label.split("_")[0] if not label.startswith("DRAMV") else "DRAM-V"
    n = sub.height
    if n == 0:
        rows.append({"tool": tool, "tier": label, "n_genes": 0})
        continue
    def ct(e):
        return int(sub.filter(e).height)
    rows.append({
        "tool": tool,
        "tier": label,
        "n_genes": n,
        "n_genomad_viral":         ct(pl.col("genomad_viral")),
        "n_strict_viral_region":   ct(pl.col("in_strict_viral_region")),
        "n_near_viral_gene":       ct(pl.col("near_viral_gene")),
        "n_near_viral_both_sides": ct(pl.col("near_viral_gene_both_sides")),
        "n_near_virus_like_mge":   ct(pl.col("near_virus_like_mge")),
        "n_near_bact_like_mge":    ct(pl.col("near_bact_like_mge")),
        "n_no_viral_evidence":     ct(~(pl.col("genomad_viral") | pl.col("in_strict_viral_region") | pl.col("near_viral_gene"))),
    })

amg_all_tier_df = pl.DataFrame(rows).with_columns([
    (pl.col("n_genomad_viral")         / pl.col("n_genes") * 100).round(1).alias("pct_genomad_viral"),
    (pl.col("n_strict_viral_region")   / pl.col("n_genes") * 100).round(1).alias("pct_strict_viral_region"),
    (pl.col("n_near_viral_gene")       / pl.col("n_genes") * 100).round(1).alias("pct_near_viral_gene"),
    (pl.col("n_near_viral_both_sides") / pl.col("n_genes") * 100).round(1).alias("pct_near_viral_both_sides"),
    (pl.col("n_near_virus_like_mge")   / pl.col("n_genes") * 100).round(1).alias("pct_near_virus_like_mge"),
    (pl.col("n_near_bact_like_mge")    / pl.col("n_genes") * 100).round(1).alias("pct_near_bact_like_mge"),
    (pl.col("n_no_viral_evidence")     / pl.col("n_genes") * 100).round(1).alias("pct_no_viral_evidence"),
])

In [54]:
amg_all_tier_df

tool,tier,n_genes,n_genomad_viral,n_strict_viral_region,n_near_viral_gene,n_near_viral_both_sides,n_near_virus_like_mge,n_near_bact_like_mge,n_no_viral_evidence,pct_genomad_viral,pct_strict_viral_region,pct_near_viral_gene,pct_near_viral_both_sides,pct_near_virus_like_mge,pct_near_bact_like_mge,pct_no_viral_evidence
str,str,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""CheckAMG""","""CheckAMG_high""",3218,3151,2886,3167,2355,1077,32,0,97.9,89.7,98.4,73.2,33.5,1.0,0.0
"""CheckAMG""","""CheckAMG_medium""",3261,3193,2912,3210,2379,1098,32,0,97.9,89.3,98.4,73.0,33.7,1.0,0.0
"""CheckAMG""","""CheckAMG_low""",3277,3205,2918,3226,2387,1107,32,0,97.8,89.0,98.4,72.8,33.8,1.0,0.0
"""DRAM-V""","""DRAMV_default""",664,621,198,526,183,33,2,2,93.5,29.8,79.2,27.6,5.0,0.3,0.3
"""DRAM-V""","""DRAMV_allow_T""",690,647,222,550,204,42,2,2,93.8,32.2,79.7,29.6,6.1,0.3,0.3
"""DRAM-V""","""DRAMV_aux_4""",783,733,287,640,255,39,2,2,93.6,36.7,81.7,32.6,5.0,0.3,0.3
"""DRAM-V""","""DRAMV_aux_4_allow_T""",4583,4477,3806,4401,3323,1693,45,2,97.7,83.0,96.0,72.5,36.9,1.0,0.0
"""VIBRANT""","""VIBRANT_high""",1270,1263,1191,1267,1092,872,16,0,99.4,93.8,99.8,86.0,68.7,1.3,0.0
"""VIBRANT""","""VIBRANT_medium""",1574,1567,1479,1569,1350,1013,22,0,99.6,94.0,99.7,85.8,64.4,1.4,0.0


## Functional annotation agreement

Within each default / high tier-set (all three tools agreeing, pairwise, or tool-alone), do the tools that annotated the gene agree on function? Per-(tool, gene, database) we take the set of `final_annot` hit ids and call agreement when two tools' sets intersect in KEGG KOs, Pfams, or any shared database.

In [55]:
def final_hit_set_by_db(frame: pl.DataFrame, tool: str, db: str, prefix: str) -> pl.DataFrame:
    return (
        frame.filter((pl.col("tool") == tool) & (pl.col("database") == db) & pl.col("final_annot"))
             .group_by("gene", "source", "sample", "ecosystem")
             .agg([
                 pl.col("hit_id").unique().alias(f"{prefix}_{db}_ids"),
                 pl.col("hit_desc").first().alias(f"{prefix}_{db}_desc"),
             ])
    )

SHARED_DBS = ["KEGG", "Pfam"]
TOOLS_NAMES = [("CheckAMG", "CheckAMG"), ("DRAM-V", "DRAMV"), ("VIBRANT", "VIBRANT")]

In [56]:
agree_frame = members
for annot_tool, prefix in TOOLS_NAMES:
    for db in SHARED_DBS:
        agree_frame = agree_frame.join(
            final_hit_set_by_db(annotations_high_default, annot_tool, db, prefix),
            on=["gene", "sample", "source", "ecosystem"], how="left",
        )
    agree_frame = agree_frame.join(
        final_hits_per_gene(annotations_high_default, annot_tool, prefix),
        on=["gene", "sample", "source", "ecosystem"], how="left",
    )

In [57]:
TIER_SETS_DH = {
    "All tools":        pl.col("CheckAMG_high") & pl.col("DRAMV_default") & pl.col("VIBRANT_high"),
    "CheckAMG & VIBRANT": pl.col("CheckAMG_high") & pl.col("VIBRANT_high") & ~pl.col("DRAMV_default"),
    "CheckAMG & DRAM-V":   pl.col("CheckAMG_high") & pl.col("DRAMV_default") & ~pl.col("VIBRANT_high"),
    "DRAM-V & VIBRANT":    pl.col("VIBRANT_high") & pl.col("DRAMV_default") & ~pl.col("CheckAMG_high"),
    "CheckAMG only":    pl.col("CheckAMG_high") & ~pl.col("DRAMV_default") & ~pl.col("VIBRANT_high"),
    "DRAM-V only":       pl.col("DRAMV_default") & ~pl.col("CheckAMG_high") & ~pl.col("VIBRANT_high"),
    "VIBRANT only":     pl.col("VIBRANT_high") & ~pl.col("CheckAMG_high") & ~pl.col("DRAMV_default"),
}

def any_list_overlap(a: str, b: str) -> pl.Expr:
    return (
        pl.when(pl.col(a).is_null() | pl.col(b).is_null())
          .then(False)
          .otherwise(pl.col(a).list.set_intersection(pl.col(b)).list.len() > 0)
    )

def agreement_counts(frame: pl.DataFrame, tier_sets: dict) -> pl.DataFrame:
    kegg_pairs = [("CheckAMG_KEGG_ids", "DRAMV_KEGG_ids"),
                  ("CheckAMG_KEGG_ids", "VIBRANT_KEGG_ids"),
                  ("DRAMV_KEGG_ids",    "VIBRANT_KEGG_ids")]
    pfam_pairs = [("CheckAMG_Pfam_ids", "DRAMV_Pfam_ids"),
                  ("CheckAMG_Pfam_ids", "VIBRANT_Pfam_ids"),
                  ("DRAMV_Pfam_ids",    "VIBRANT_Pfam_ids")]
    any_pairs = [("CheckAMG_final_hit_ids", "DRAMV_final_hit_ids"),
                 ("CheckAMG_final_hit_ids", "VIBRANT_final_hit_ids"),
                 ("DRAMV_final_hit_ids",    "VIBRANT_final_hit_ids")]
    rows = []
    for name, expr in tier_sets.items():
        sub = frame.filter(expr)
        n = sub.height
        if n == 0:
            rows.append({"set": name, "n_genes": 0,
                         "n_any_KEGG_agree": 0, "n_any_Pfam_agree": 0, "n_any_hit_agree": 0,
                         "pct_any_KEGG_agree": 0.0, "pct_any_Pfam_agree": 0.0, "pct_any_hit_agree": 0.0})
            continue
        n_kegg = int(sub.filter(pl.any_horizontal([any_list_overlap(a, b) for a, b in kegg_pairs])).height)
        n_pfam = int(sub.filter(pl.any_horizontal([any_list_overlap(a, b) for a, b in pfam_pairs])).height)
        n_any  = int(sub.filter(pl.any_horizontal([any_list_overlap(a, b) for a, b in any_pairs])).height)
        rows.append({
            "set": name, "n_genes": n,
            "n_any_KEGG_agree": n_kegg, "n_any_Pfam_agree": n_pfam, "n_any_hit_agree": n_any,
            "pct_any_KEGG_agree": round(100 * n_kegg / n, 2),
            "pct_any_Pfam_agree": round(100 * n_pfam / n, 2),
            "pct_any_hit_agree":  round(100 * n_any  / n, 2),
        })
    return pl.DataFrame(rows)

In [58]:
agreement_df = agreement_counts(agree_frame, TIER_SETS_DH)

In [59]:
agreement_df

set,n_genes,n_any_KEGG_agree,n_any_Pfam_agree,n_any_hit_agree,pct_any_KEGG_agree,pct_any_Pfam_agree,pct_any_hit_agree
str,i64,i64,i64,i64,f64,f64,f64
"""All tools""",0,0,0,0,0.0,0.0,0.0
"""CheckAMG & VIBRANT""",221,175,173,216,79.19,78.28,97.74
"""CheckAMG & DRAM-V""",0,0,0,0,0.0,0.0,0.0
"""DRAM-V & VIBRANT""",0,0,0,0,0.0,0.0,0.0
"""CheckAMG only""",2997,813,1121,1347,27.13,37.4,44.94
"""DRAM-V only""",664,6,5,8,0.9,0.75,1.2
"""VIBRANT only""",1049,458,561,659,43.66,53.48,62.82


## Reference-database heterogeneity

Each tool calls AMGs only from its own curated list of gene identifiers. CheckAMG requires an AMG weight of at least 0.6 by default. The weight reflects the share of metabolic pathways an HMM is associated with and its VL-score, so it can also be matched to many of the other tools' AMG identifiers.

In [60]:
CHECKAMG_DEFAULT_WEIGHT = 0.6

def strip_pfam_version(expr: pl.Expr) -> pl.Expr:
    return expr.cast(pl.Utf8).str.split(".").list.first()

In [61]:
amgs = pl.read_csv("../CheckAMG/files/AMGs.tsv", separator="\t")
amg_filters = pl.read_csv("../CheckAMG/files/AMG_filters.tsv", separator="\t")
amg_database = pl.read_csv("./data/amg_database.tsv", separator="\t")
vibrant_amgs = (
    pl.read_csv("./data/VIBRANT_AMGs.tsv", separator="\t", has_header=False, new_columns=["ko"])
      .filter(pl.col("ko") != "KO").unique()
)

In [62]:
print(f"AMGs.tsv: {amgs.height:,}")
print(f"AMG_filters.tsv: {amg_filters.height:,}")
print(f"amg_database.tsv (DRAM-V): {amg_database.height:,}")
print(f"VIBRANT_AMGs.tsv: {vibrant_amgs.height:,}")

AMGs.tsv: 60,770
AMG_filters.tsv: 38,768
amg_database.tsv (DRAM-V): 279
VIBRANT_AMGs.tsv: 2,826


In [63]:
dramv_ko = amg_database.filter(pl.col("KO").is_not_null()).select(pl.col("KO").alias("ko")).unique()
dramv_pfam = (
    amg_database.filter(pl.col("PFAM").is_not_null())
                .select(pl.col("PFAM").str.split("; ").alias("pfam_list"))
                .explode("pfam_list")
                .with_columns(strip_pfam_version(pl.col("pfam_list")).alias("pfam"))
                .filter(pl.col("pfam").str.starts_with("PF"))
                .select("pfam").unique()
)

# Sorted because unique() does not preserve order, which reordered the appended database columns on every run
checkamg_dbs = sorted(amgs.select(pl.col("db")).filter(pl.col("db").is_not_null()).unique()["db"].to_list())
amg_super = (
    pl.concat([
        dramv_pfam.rename({"pfam": "HMM"}),
        dramv_ko.rename({"ko": "HMM"}),
        vibrant_amgs.rename({"ko": "HMM"}),
        (
            amgs
            .select("id")
            .with_columns(strip_pfam_version(pl.col("id")).alias("id"))
            .rename({"id": "HMM"})
        )
    ])
    .with_columns([
        pl.when(
            pl.col("HMM")
            .is_in(
                amgs
                .filter(pl.col("db")==db)
                .select("id")
                .with_columns(strip_pfam_version(pl.col("id")).alias("id"))
                ["id"].to_list()
            )
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias(db)
        for db in checkamg_dbs
    ])
    .with_columns([
        pl.when(pl.col("HMM").is_in(dramv_pfam["pfam"].to_list()))
        .then(pl.lit(True))
        .otherwise(pl.col("Pfam"))
        .alias("Pfam"),
        pl.when((pl.col("HMM").is_in(dramv_ko["ko"].to_list())) | (pl.col("HMM").is_in(vibrant_amgs["ko"].to_list())))
        .then(pl.lit(True))
        .otherwise(pl.col("KEGG"))
        .alias("KEGG"),
    ])
    .filter(pl.col("HMM").is_not_null())
    .unique()
    .sort("HMM")
)
amg_super

HMM,CAMPER,FOAM,KEGG,METABOLIC,Pfam,dbCAN
str,bool,bool,bool,bool,bool,bool
"""AA1""",true,false,false,false,false,true
"""AA10""",false,false,false,false,false,true
"""AA13""",false,false,false,false,false,true
"""AA14""",false,false,false,false,false,true
"""AA15""",false,false,false,false,false,true
…,…,…,…,…,…,…
"""soxZ""",false,false,false,true,false,false
"""sulfide_quinone_oxidoreductase_sqr""",false,false,false,true,false,false
"""sulfocyanin""",false,false,false,true,false,false


In [64]:
upset_amg_ref_combined = (
    amg_super
    .with_columns([
        pl.when(
            pl.col("HMM").is_in(
                amgs
                .filter(pl.col("AMG_weight")>=CHECKAMG_DEFAULT_WEIGHT)
                .with_columns(strip_pfam_version(pl.col("id")).alias("id"))["id"].to_list()
            )
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias(f"CheckAMG (AMG weight >= {CHECKAMG_DEFAULT_WEIGHT})"),

        pl.when(
            pl.col("HMM").is_in(
                amgs
                .with_columns(strip_pfam_version(pl.col("id")).alias("id"))["id"].to_list()
            )
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("CheckAMG (all)"),

        pl.when((pl.col("HMM").is_in(dramv_pfam["pfam"].to_list())) | (pl.col("HMM").is_in(dramv_ko["ko"].to_list())))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("DRAMV"),

        pl.when(pl.col("HMM").is_in(vibrant_amgs["ko"].to_list()))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias("VIBRANT")
    ])
)
upset_amg_ref_combined

HMM,CAMPER,FOAM,KEGG,METABOLIC,Pfam,dbCAN,CheckAMG (AMG weight >= 0.6),CheckAMG (all),DRAMV,VIBRANT
str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""AA1""",true,false,false,false,false,true,true,true,false,false
"""AA10""",false,false,false,false,false,true,true,true,false,false
"""AA13""",false,false,false,false,false,true,true,true,false,false
"""AA14""",false,false,false,false,false,true,true,true,false,false
"""AA15""",false,false,false,false,false,true,true,true,false,false
…,…,…,…,…,…,…,…,…,…,…
"""soxZ""",false,false,false,true,false,false,true,true,false,false
"""sulfide_quinone_oxidoreductase_sqr""",false,false,false,true,false,false,true,true,false,false
"""sulfocyanin""",false,false,false,true,false,false,true,true,false,false


In [65]:
ref_summary = pl.DataFrame([
    {"tool": "CheckAMG", "space": "KO",   "level": "any AMG weight",   "n": upset_amg_ref_combined.filter((pl.col("CheckAMG (all)")) & (pl.col("KEGG"))).height},
    {"tool": "CheckAMG", "space": "KO",   "level": f"AMG weight >= {CHECKAMG_DEFAULT_WEIGHT}", "n": upset_amg_ref_combined.filter((pl.col(f"CheckAMG (AMG weight >= {CHECKAMG_DEFAULT_WEIGHT})")) & (pl.col("KEGG"))).height},
    {"tool": "CheckAMG", "space": "Pfam", "level": "any AMG weight",   "n": upset_amg_ref_combined.filter((pl.col("CheckAMG (all)")) & (pl.col("Pfam"))).height},
    {"tool": "CheckAMG", "space": "Pfam", "level": f"AMG weight >= {CHECKAMG_DEFAULT_WEIGHT}", "n": upset_amg_ref_combined.filter((pl.col(f"CheckAMG (AMG weight >= {CHECKAMG_DEFAULT_WEIGHT})")) & (pl.col("Pfam"))).height},
    {"tool": "CheckAMG", "space": "all",  "level": "any AMG weight",   "n": upset_amg_ref_combined.filter((pl.col("CheckAMG (all)"))).height},
    {"tool": "CheckAMG", "space": "all",  "level": f"AMG weight >= {CHECKAMG_DEFAULT_WEIGHT}", "n": int(upset_amg_ref_combined.filter(pl.col(f"CheckAMG (AMG weight >= {CHECKAMG_DEFAULT_WEIGHT})")).height)},
    {"tool": "DRAMV",    "space": "KO",   "level": "curated",          "n": upset_amg_ref_combined.filter((pl.col("DRAMV")) & (pl.col("KEGG"))).height},
    {"tool": "DRAMV",    "space": "Pfam", "level": "curated",          "n": upset_amg_ref_combined.filter((pl.col("DRAMV")) & (pl.col("Pfam"))).height},
    {"tool": "VIBRANT",  "space": "KO",   "level": "curated",          "n": upset_amg_ref_combined.filter((pl.col("VIBRANT")) & (pl.col("KEGG"))).height},
])

In [66]:
ref_summary

tool,space,level,n
str,str,str,i64
"""CheckAMG""","""KO""","""any AMG weight""",10055
"""CheckAMG""","""KO""","""AMG weight >= 0.6""",8633
"""CheckAMG""","""Pfam""","""any AMG weight""",1611
"""CheckAMG""","""Pfam""","""AMG weight >= 0.6""",1069
"""CheckAMG""","""all""","""any AMG weight""",60311
"""CheckAMG""","""all""","""AMG weight >= 0.6""",55999
"""DRAMV""","""KO""","""curated""",34
"""DRAMV""","""Pfam""","""curated""",251
"""VIBRANT""","""KO""","""curated""",2826


Some HMMs occur in multiple databases (e.g., CAZymes and KOs in both dbCAN and CAMPER, Pfams in both Pfam and METABOLIC, etc.), which means that they sometimes appear in CheckAMG's database more than once. This is why the number of *unique* HMMs in the entire CheckAMG AMG reference is less than the number of HMMs (rows) in the CheckAMG reference AMG table.

In [67]:
upset_amg_ref_combined.write_csv(OUTPUT_TABLES_DIR.joinpath("AMG_reference_overlap_combined.tsv"), separator="\t")


### CheckAMG AMG weight distribution per database

In [68]:
checkamg_amg_weights_by_db = (
    amgs.with_columns(
        pl.when(pl.col("AMG_weight") >= 0.8).then(pl.lit("very_high"))
          .when(pl.col("AMG_weight") >= 0.6).then(pl.lit("high"))
          .when(pl.col("AMG_weight") >= 0.4).then(pl.lit("medium"))
          .when(pl.col("AMG_weight") >= 0.2).then(pl.lit("low"))
          .otherwise(pl.lit("very_low"))
          .alias("bin")
    )
    .group_by(["db", "bin"]).agg(pl.len().alias("n"))
    .sort(["db", "bin"])
)

In [69]:
with pl.Config(tbl_rows=100, tbl_width_chars=500):
    print(checkamg_amg_weights_by_db)

shape: (29, 3)
┌───────────┬───────────┬───────┐
│ db        ┆ bin       ┆ n     │
│ ---       ┆ ---       ┆ ---   │
│ str       ┆ str       ┆ u64   │
╞═══════════╪═══════════╪═══════╡
│ null      ┆ medium    ┆ 1     │
│ null      ┆ very_high ┆ 56    │
│ CAMPER    ┆ high      ┆ 51    │
│ CAMPER    ┆ medium    ┆ 1     │
│ CAMPER    ┆ very_high ┆ 227   │
│ FOAM      ┆ high      ┆ 21099 │
│ FOAM      ┆ low       ┆ 116   │
│ FOAM      ┆ medium    ┆ 1939  │
│ FOAM      ┆ very_high ┆ 24957 │
│ FOAM      ┆ very_low  ┆ 13    │
│ KEGG      ┆ high      ┆ 1759  │
│ KEGG      ┆ low       ┆ 445   │
│ KEGG      ┆ medium    ┆ 906   │
│ KEGG      ┆ very_high ┆ 6874  │
│ KEGG      ┆ very_low  ┆ 71    │
│ METABOLIC ┆ high      ┆ 44    │
│ METABOLIC ┆ low       ┆ 3     │
│ METABOLIC ┆ medium    ┆ 20    │
│ METABOLIC ┆ very_high ┆ 230   │
│ METABOLIC ┆ very_low  ┆ 1     │
│ Pfam      ┆ high      ┆ 154   │
│ Pfam      ┆ low       ┆ 138   │
│ Pfam      ┆ medium    ┆ 404   │
│ Pfam      ┆ very_high ┆ 915   │

## Likely AMG vs weak AMG per call

For every AMG-called gene, look up each tool's own `final_annot` hit ids in CheckAMG's `AMGs.tsv` reference and use the associated `AMG_weight` as a continuous, tool-agnostic rubric. The per-gene best weight is the MAX `AMG_weight` across the tool's final hits. The CheckAMG `AMG_filters.tsv` categories (essential / glucan / lipid / methyl / nucleotide) are also OR-aggregated across the tool's final hits and reported separately. They are reported rather than used as a filter, because nucleotide-metabolism genes with `AMG_weight >= 0.8`, for example, are still likely AMGs.

In [70]:
FILTER_CATEGORIES = ["filter_essential", "filter_glucan", "filter_lipid", "filter_methyl", "filter_nucleotide"]
CHECKAMG_HIGH_WEIGHT = 0.8
CHECKAMG_LOW_WEIGHT  = 0.4

WEIGHT_BIN_ORDER = [
    "weight_ge_0.8", "weight_0.6_0.8", "weight_0.4_0.6", "weight_lt_0.4",
    "no_reference", "no_annotation", "not_called",
]
CATEGORY_ORDER = ["methyl", "nucleotide", "glucan", "lipid", "essential", "none"]

Reference tables reduced to (db, id_key) lookups, with Pfam versions stripped.

In [71]:
amgs_slim = (
    amgs.with_columns(
        pl.when(pl.col("db") == "Pfam")
          .then(pl.col("id").str.split(".").list.first())
          .otherwise(pl.col("id"))
          .alias("id_key")
    )
    .group_by(["db", "id_key"])
    .agg([pl.col("AMG_weight").max().alias("AMG_weight"), pl.col("amg_level").first().alias("amg_level")])
)
filters_slim = (
    amg_filters.with_columns(
        pl.when(pl.col("db") == "Pfam")
          .then(pl.col("id").str.split(".").list.first())
          .otherwise(pl.col("id"))
          .alias("id_key")
    )
    .group_by(["db", "id_key"])
    .agg([pl.col(c).any().alias(c) for c in FILTER_CATEGORIES])
)

Attach the reference weight and filter flags to every final annotation, then collapse to one row per tool and gene, taking the maximum weight and OR-ing the filter flags.

In [72]:
final_annots = (
    annotations_high_default.filter(pl.col("final_annot"))
    .with_columns(
        pl.when(pl.col("database") == "Pfam")
          .then(pl.col("hit_id").str.split(".").list.first())
          .otherwise(pl.col("hit_id"))
          .alias("id_key"),
    )
    .join(amgs_slim, left_on=["database", "id_key"], right_on=["db", "id_key"], how="left")
    .join(filters_slim, left_on=["database", "id_key"], right_on=["db", "id_key"], how="left")
    .with_columns([pl.col(c).fill_null(False) for c in FILTER_CATEGORIES])
)

per_tool_gene = (
    final_annots.group_by(["sample", "source", "ecosystem", "tool", "gene"])
    .agg([
        pl.col("AMG_weight").max().alias("best_AMG_weight"),
        pl.col("AMG_weight").is_not_null().any().alias("in_amg_ref"),
        pl.col("hit_id").is_not_null().any().alias("has_hit"),
    ] + [pl.col(c).any().alias(c) for c in FILTER_CATEGORIES])
    .with_columns([
        pl.when(~pl.col("has_hit"))
          .then(pl.lit("no_annotation"))
          .when(~pl.col("in_amg_ref"))
          .then(pl.lit("no_reference"))
          .when(pl.col("best_AMG_weight") >= CHECKAMG_HIGH_WEIGHT).then(pl.lit("weight_ge_0.8"))
          .when(pl.col("best_AMG_weight") >= CHECKAMG_DEFAULT_WEIGHT).then(pl.lit("weight_0.6_0.8"))
          .when(pl.col("best_AMG_weight") >= CHECKAMG_LOW_WEIGHT).then(pl.lit("weight_0.4_0.6"))
          .otherwise(pl.lit("weight_lt_0.4"))
          .alias("weight_bin"),
        pl.when(pl.col("filter_essential")).then(pl.lit("essential"))
          .when(pl.col("filter_nucleotide")).then(pl.lit("nucleotide"))
          .when(pl.col("filter_glucan")).then(pl.lit("glucan"))
          .when(pl.col("filter_lipid")).then(pl.lit("lipid"))
          .when(pl.col("filter_methyl")).then(pl.lit("methyl"))
          .otherwise(pl.lit("none"))
          .alias("dominant_category"),
    ])
)

In [73]:
per_tool_gene

sample,source,ecosystem,tool,gene,best_AMG_weight,in_amg_ref,has_hit,filter_essential,filter_glucan,filter_lipid,filter_methyl,filter_nucleotide,weight_bin,dominant_category
str,str,str,str,str,f64,bool,bool,bool,bool,bool,bool,bool,str,str
"""soil_T42_15_2_56""","""Mixed metagenomes""","""Soil""","""CheckAMG""","""scaffold_1108_c1_15""",0.637947,true,true,false,false,false,true,false,"""weight_0.6_0.8""","""methyl"""
"""soil_T42_15_3_57""","""Mixed metagenomes""","""Soil""","""CheckAMG""","""scaffold_881_c1_15""",0.637947,true,true,false,false,false,true,false,"""weight_0.6_0.8""","""methyl"""
"""virus_genomes_gut""","""Viral genomes""","""Human gut""","""CheckAMG""","""IMGVR_UViG_3300045988_160735_3300045988_Ga0495776_141072_117""",null,false,true,true,false,false,false,false,"""no_reference""","""essential"""
"""soil_T42_15_3_57""","""Mixed metagenomes""","""Soil""","""CheckAMG""","""scaffold_858_c1_33""",0.825603,true,true,false,false,false,false,false,"""weight_ge_0.8""","""none"""
"""freshwater_Ga0485159_contigs""","""Mixed metagenomes""","""Aquatic""","""CheckAMG""","""Ga0485159_0000080_43""",null,false,true,false,false,false,false,true,"""no_reference""","""nucleotide"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""virus_genomes_soil""","""Viral genomes""","""Soil""","""CheckAMG""","""IMGVR_UViG_2823053753_000001_2823053753_2823053778_40""",0.504684,true,true,true,false,false,true,true,"""weight_0.4_0.6""","""essential"""
"""freshwater_Ga0485175_contigs""","""Viromes""","""Aquatic""","""DRAM-V""","""Ga0485175_0018000_6""",0.5,true,true,false,false,false,false,false,"""weight_0.4_0.6""","""none"""
"""freshwater_Ga0485174_contigs""","""Viromes""","""Aquatic""","""VIBRANT""","""Ga0485174_0000229_50""",0.438619,true,true,false,true,false,false,true,"""weight_0.4_0.6""","""nucleotide"""


Pivot to one row per gene, with weight bin, dominant category, and filter flags for each tool.

In [74]:
tool_keys = [("CheckAMG", "CheckAMG"), ("DRAM-V", "DRAMV"), ("VIBRANT", "VIBRANT")]
members_weights = members_annot
for annot_tool, prefix in tool_keys:
    tf = (
        per_tool_gene.filter(pl.col("tool") == annot_tool).drop("tool")
                     .rename({c: f"{prefix}_{c}" for c in per_tool_gene.columns if c not in ("tool", "gene", "source", "sample", "ecosystem")})
    )
    members_weights = members_weights.join(tf, on=["gene", "sample", "source", "ecosystem"], how="left")

for _, prefix in tool_keys:
    members_weights = members_weights.with_columns([
        pl.col(f"{prefix}_weight_bin").fill_null("not_called"),
        pl.col(f"{prefix}_dominant_category").fill_null("none"),
    ] + [pl.col(f"{prefix}_{c}").fill_null(False) for c in FILTER_CATEGORIES])

AMG weights and categories for each gene, annotation, and tool.

In [75]:
amg_predictions_categorized = (
    per_tool_gene
    .select([
        "gene", "source", "sample", "ecosystem", "tool",
    ])
    .join(
        annotations,
        on=["source", "sample", "ecosystem", "gene", "tool"],
        how="left"
    )
    .filter(pl.col("final_annot")) # restrict to final annotations reported by each tool
    .join(
        amgs_slim,
        left_on=["database", "hit_id"],
        right_on=["db", "id_key"],
        how="left",
    )
    .join(
        filters_slim,
        left_on=["database", "hit_id"],
        right_on=["db", "id_key"],
        how="left",
    )
    .join(
        members,
        on=["source", "sample", "ecosystem", "gene"],
        how="left",
    )
    .with_columns(
        pl.any_horizontal(
            pl.col("filter_essential").is_not_null(),
            pl.col("filter_glucan").is_not_null(),
            pl.col("filter_lipid").is_not_null(),
            pl.col("filter_methyl").is_not_null(),
            pl.col("filter_nucleotide").is_not_null(),
        ).alias("has_filter")
    )
    .sort(
        ["tool",  "gene", "has_filter"],
        descending=[False, False, True],
    )
    .drop("has_filter")
)

In [76]:
amg_predictions_categorized

gene,source,sample,ecosystem,tool,database,hit_id,hit_desc,bitscore,evalue,final_annot,AMG_weight,amg_level,filter_essential,filter_glucan,filter_lipid,filter_methyl,filter_nucleotide,scaffold,genomad_viral,in_strict_viral_region,viral_gene_left_dist,viral_gene_right_dist,MGE_gene_left_dist,MGE_gene_right_dist,MGE_gene_left_V_score,MGE_gene_right_V_score,CheckAMG_high,CheckAMG_medium,CheckAMG_low,DRAMV_default,DRAMV_allow_T,DRAMV_aux_4,DRAMV_aux_4_allow_T,VIBRANT_high,VIBRANT_medium,VIBRANT_low,DRAMV_auxiliary_score,DRAMV_M,DRAMV_T,DRAMV_V,DRAMV_A,DRAMV_P,DRAMV_B,DRAMV_F
str,str,str,str,str,str,str,str,f64,f64,bool,f64,str,bool,bool,bool,bool,bool,str,bool,bool,f32,f32,f32,f32,f32,f32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64,bool,bool,bool,bool,bool,bool,bool
"""Ga0485157_0000020_8""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""Pfam""","""PF01764""","""Lipase (class 3)""",100.152611,1.2137e-27,true,0.699204,"""high""",null,null,null,null,null,"""Ga0485157_0000020""",false,true,237.0,13092.0,null,null,null,null,true,true,true,false,false,false,false,false,false,false,null,false,false,false,false,false,false,false
"""Ga0485157_0000020_8""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""PHROG""","""phrog_21992""","""esterase/lipase""",72.468353,5.0647e-19,true,null,null,null,null,null,null,null,"""Ga0485157_0000020""",false,true,237.0,13092.0,null,null,null,null,true,true,true,false,false,false,false,false,false,false,null,false,false,false,false,false,false,false
"""Ga0485157_0000095_62""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""Pfam""","""PF00676""","""Dehydrogenase E1 component""",63.078701,2.7360e-16,true,0.763093,"""high""",null,null,null,null,null,"""Ga0485157_0000095""",true,true,2910.0,1179.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_62""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""FOAM""","""HMMsoil93075""","""PDHA, pdhA; pyruvate dehydrogenase E1 component alpha subunit [EC:1.2.4.1]; PDHB, pdhB; pyruvate deh…",115.909142,2.4009e-32,true,0.763773,"""high""",null,null,null,null,null,"""Ga0485157_0000095""",true,true,2910.0,1179.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
"""Ga0485157_0000095_62""","""Mixed metagenomes""","""freshwater_Ga0485157_contigs""","""Aquatic""","""CheckAMG""","""PHROG""","""phrog_34187""",null,178.282883,2.7760e-51,true,null,null,null,null,null,null,null,"""Ga0485157_0000095""",true,true,2910.0,1179.0,null,null,null,null,true,true,true,false,false,false,true,false,false,false,4,true,true,false,false,false,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""scaffold_9_c1_92""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","""VIBRANT""","""KEGG""","""K06920""","""queC; 7-cyano-7-deazaguanine synthase [EC:6.3.4.20]""",216.1,9.8000e-64,true,0.468197,"""medium""",true,false,false,false,true,"""scaffold_9_c1""",true,true,3573.0,10593.0,null,32496.0,null,10.0,false,false,false,false,false,false,true,true,true,true,4,true,true,false,false,false,true,false
"""scaffold_9_c1_92""","""Mixed metagenomes""","""soil_T42_15_3_57""","""Soil""","""VIBRANT""","""Pfam""","""PF06508""","""Queuosine biosynthesis protein QueC""",178.7,1.4000e-52,true,null,null,null,null,null,null,null,"""scaffold_9_c1""",true,true,3573.0,10593.0,null,32496.0,null,10.0,false,false,false,false,false,false,true,true,true,true,4,true,true,false,false,false,true,false
"""scaffold_9_c1_98""","""Mixed metagenomes""","""soil_T42_15_2_56""","""Soil""","""VIBRANT""","""KEGG""","""K06211""","""nadR; HTH-type transcriptional regulator, transcriptional repressor of NAD biosynthesis genes [EC:2.…",123.5,2.3000e-35,true,n

In [77]:
amg_predictions_categorized.write_csv(OUTPUT_TABLES_DIR.joinpath("AMG_annotations_categorized.tsv"), separator="\t")

### Top functions per tier-set

In [78]:
GENE_KEY_TF = ["gene", "sample", "source", "ecosystem"]


def _normalize_tool(name: str) -> str:
    return name.strip().replace("-", "").upper()

_TOOL_NORM_MAP = {
    _normalize_tool("CheckAMG"): "CheckAMG",
    _normalize_tool("DRAMV"):    "DRAM-V",
    _normalize_tool("VIBRANT"):  "VIBRANT",
}

def _tools_from_label(set_label: str) -> list[str]:
    """
    Extract normalized tool column values from a set label.
    'CheckAMG only'        -> ['CheckAMG']
    'CheckAMG & VIBRANT'   -> ['CheckAMG', 'VIBRANT']
    'DRAM-V & VIBRANT'     -> ['DRAMV', 'VIBRANT']
    'All tools'            -> []  (no tool filter)
    """
    label = set_label.replace(" only", "")
    if "&" in label:
        parts = [t.strip() for t in label.split("&")]
    elif label.strip() == "All tools":
        return []
    else:
        parts = [label.strip()]
    return [_TOOL_NORM_MAP[_normalize_tool(p)] for p in parts if _normalize_tool(p) in _TOOL_NORM_MAP]

def _best_hit_per_gene(
    sub: pl.DataFrame,
    tool_values: list[str],
) -> pl.DataFrame:
    valid = sub.filter(
        pl.col("hit_desc").is_not_null()
        & (pl.col("hit_desc").str.strip_chars() != "")
    )

    def _pick_best(df: pl.DataFrame) -> pl.DataFrame:
        # Best hit: highest AMG_weight, then bitscore and e-value, with hit_id and the longer description as reproducible tie-breaks
        # Tool is not a sort key because alphabetical order would resolve every tie in CheckAMG's favor
        # The key is the four gene-identifying columns, since a gene name repeats across samples.
        df = df.with_columns(pl.col("hit_desc").str.len_chars().alias("_desc_len"))
        has_weight = df.filter(pl.col("AMG_weight").is_not_null())
        genes_with_weight = has_weight.select(GENE_KEY_TF).unique()
        without_weight = df.join(genes_with_weight, on=GENE_KEY_TF, how="anti")
        return pl.concat([
            has_weight.sort(["AMG_weight", "bitscore", "evalue", "hit_id", "_desc_len", "hit_desc"],
                            descending=[True, True, False, False, True, False], maintain_order=True)
                    .unique(subset=GENE_KEY_TF, keep="first", maintain_order=True),
            without_weight.sort(["bitscore", "evalue", "hit_id", "_desc_len", "hit_desc"],
                                descending=[True, False, False, True, False], maintain_order=True)
                        .unique(subset=GENE_KEY_TF, keep="first", maintain_order=True),
        ]).select(GENE_KEY_TF + ["hit_id", "hit_desc", "bitscore", "evalue"])

    if len(tool_values) < 2:
        pool = valid.filter(pl.col("tool").is_in(tool_values)) if tool_values else valid
        return _pick_best(pool)

    per_tool_hits = [
        valid.filter(pl.col("tool") == tv).select(GENE_KEY_TF + ["hit_id"]).unique()
        for tv in tool_values
    ]
    shared = per_tool_hits[0]
    for other in per_tool_hits[1:]:
        shared = shared.join(other, on=GENE_KEY_TF + ["hit_id"], how="inner")

    if shared.height > 0:
        shared_rows = _pick_best(valid.join(shared, on=GENE_KEY_TF + ["hit_id"], how="inner"))
    else:
        shared_rows = pl.DataFrame(
            {c: [] for c in GENE_KEY_TF + ["hit_id", "hit_desc", "bitscore", "evalue"]},
            schema={
                **{c: pl.Utf8 for c in GENE_KEY_TF},
                "hit_id": pl.Utf8,
                "hit_desc": pl.Utf8,
                "bitscore": pl.Float64,
                "evalue": pl.Float64,
            }
        )

    fallback_rows = _pick_best(
        valid
        .filter(pl.col("tool").is_in(tool_values))
        .join(shared_rows.select(GENE_KEY_TF), on=GENE_KEY_TF, how="anti")
    )

    return pl.concat([shared_rows, fallback_rows])

def hit_set_composition(
    amg_predictions_categorizes: pl.DataFrame,
    tier_sets: dict[str, pl.Expr],
) -> pl.DataFrame:
    dfs = []
    for set_label, set_expr in tier_sets.items():
        label_tools = _tools_from_label(set_label)
        tool_values = [_TOOL_NORM_MAP[_normalize_tool(t)] for t in label_tools
                       if _normalize_tool(t) in _TOOL_NORM_MAP]

        sub = amg_predictions_categorizes.filter(set_expr)
        best_per_gene = _best_hit_per_gene(sub, tool_values)

        if best_per_gene.height == 0:
            continue

        agg = (
            best_per_gene
            .group_by("hit_id")
            .agg([
                pl.concat_str(GENE_KEY_TF, separator="\x1f").n_unique().alias("n"),
                # The representative description follows the same rule as the row selection
                pl.col("hit_desc").sort_by([pl.col("hit_desc").str.len_chars(), pl.col("hit_desc")],
                                           descending=[True, False]).first().alias("hit_desc"),
                pl.col("bitscore").median().alias("median_bitscore"),
                pl.col("evalue").median().alias("median_evalue"),
            ])
            .with_columns(pl.lit(set_label).alias("set"))
        )
        dfs.append(agg)

    return (
        pl.concat(dfs)
        .select(["hit_id", "hit_desc", "n", "median_bitscore", "median_evalue", "set"])
        .sort(["n", "set", "hit_id"], descending=[True, False, False])
    )

In [79]:
hits_set_comp = hit_set_composition(amg_predictions_categorized, TIER_SETS_DH)

In [80]:
with pl.Config(tbl_rows=100, tbl_width_chars=500):
    print(hits_set_comp)

shape: (694, 6)
┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬─────┬─────────────────┬───────────────┬────────────────────┐
│ hit_id        ┆ hit_desc                                                                                              ┆ n   ┆ median_bitscore ┆ median_evalue ┆ set                │
│ ---           ┆ ---                                                                                                   ┆ --- ┆ ---             ┆ ---           ┆ ---                │
│ str           ┆ str                                                                                                   ┆ u64 ┆ f64             ┆ f64           ┆ str                │
╞═══════════════╪═══════════════════════════════════════════════════════════════════════════════════════════════════════╪═════╪═════════════════╪═══════════════╪════════════════════╡
│ K07336        ┆ K07336; PKHD-type hydroxylase [EC:1.14.11.-]       

### Weight composition at default / high

In [81]:
def _assign_none_categories(df: pl.DataFrame, prefix: str) -> pl.DataFrame:
    cat_col = f"{prefix}_dominant_category"
    w_col = f"{prefix}_best_AMG_weight"

    return df.with_columns(
        pl.when(
            pl.col(cat_col).is_null() |
            (pl.col(cat_col) == "none")
        )
        .then(
            pl.when(pl.col(w_col).is_null()).then(pl.lit("none_no_weight"))
            .when(pl.col(w_col) >= 0.6).then(pl.lit("none_likely"))
            .otherwise(pl.lit("none_unlikely"))
        )
        .otherwise(pl.col(cat_col))
        .alias(cat_col)
    )

def weight_composition(m: pl.DataFrame, tier_cols: list[tuple[str, str]]) -> pl.DataFrame:
    rows = []
    for tier, prefix in tier_cols:
        sub = m.filter(pl.col(tier))
        n_tier = sub.height
        for b in WEIGHT_BIN_ORDER:
            n = int(sub.filter(pl.col(f"{prefix}_weight_bin") == b).height)
            rows.append({"tier": tier, "tool": prefix, "weight_bin": b,
                         "n": n, "n_tier": n_tier,
                         "pct": round(100 * n / n_tier, 2) if n_tier else 0.0})
    return pl.DataFrame(rows).filter(pl.col("weight_bin") != "none")

def category_composition(m: pl.DataFrame, tier_cols: list[tuple[str, str]]) -> pl.DataFrame:
    rows = []

    for tier, prefix in tier_cols:
        sub = m.filter(pl.col(tier))
        sub = _assign_none_categories(sub, prefix)

        n_tier = sub.height

        categories = CATEGORY_ORDER + ["none_likely", "none_unlikely", "none_no_weight"]

        for c in categories:
            n = int(sub.filter(pl.col(f"{prefix}_dominant_category") == c).height)
            rows.append({
                "tier": tier,
                "tool": prefix,
                "category": c,
                "n": n,
                "n_tier": n_tier,
                "pct": round(100 * n / n_tier, 2) if n_tier else 0.0
            })

    return pl.DataFrame(rows).filter(pl.col("category") != "none")

def category_weight_composition(m: pl.DataFrame, tier_cols: list[tuple[str, str]]) -> pl.DataFrame:
    dfs = []

    categories = CATEGORY_ORDER + ["none_likely", "none_unlikely", "none_no_weight"]

    full_grid = (
        pl.DataFrame({"category": categories})
        .join(pl.DataFrame({"weight_bin": WEIGHT_BIN_ORDER}), how="cross")
    )

    for tier, prefix in tier_cols:
        sub = m.filter(pl.col(tier))
        sub = _assign_none_categories(sub, prefix)

        n_tier = sub.height
        if n_tier == 0:
            continue

        counts = (
            sub
            .group_by([f"{prefix}_dominant_category", f"{prefix}_weight_bin"])
            .agg(pl.len().alias("n"))
            .rename({
                f"{prefix}_dominant_category": "category",
                f"{prefix}_weight_bin": "weight_bin"
            })
        )

        out = (
            full_grid
            .join(counts, on=["category", "weight_bin"], how="left")
            .with_columns([
                pl.col("n").fill_null(0),
                pl.lit(tier).alias("tier"),
                pl.lit(prefix).alias("tool"),
                pl.lit(n_tier).alias("n_tier")
            ])
            .with_columns(
                (100 * pl.col("n") / pl.col("n_tier")).round(2).alias("pct")
            )
        )

        dfs.append(out)

    return (
        pl.concat(dfs)
        .select(["tier", "tool", "category", "weight_bin", "n", "n_tier", "pct"])
        .sort(["tier", "tool", "category", "weight_bin"], descending=[False, False, False, True])
        .filter(pl.col("weight_bin") != "none")
        .filter(pl.col("category") != "none")
    )

def source_db_composition(
    m: pl.DataFrame,
    tier_cols: list[tuple[str, str]]
) -> pl.DataFrame:
    rows = []

    for tier_col, tool in tier_cols:
        if tool == "DRAMV":
            tool = "DRAM-V"
        sub = m.filter(
            (pl.col("tool") == tool) &
            pl.col(tier_col)
        )

        for db in checkamg_dbs:
            n = sub.filter(pl.col("database") == db).height
            rows.append({
                "tier": tier_col,
                "tool": tool,
                "database": db,
                "n": n
            })

    result = pl.DataFrame(rows)

    return (
        result
        .with_columns(
            pl.col("n")
            .sum()
            .over(["tier", "tool"])
            .alias("n_tier")
        )
        .with_columns(
            (100 * pl.col("n") / pl.col("n_tier"))
            .round(2)
            .alias("pct")
        )
    )

In [82]:
DEFAULT_HIGH_WITH_PREFIX = [
    ("CheckAMG_high", "CheckAMG"),
    ("DRAMV_default", "DRAMV"),
    ("VIBRANT_high",  "VIBRANT"),
]

weight_default_high = weight_composition(members_weights, DEFAULT_HIGH_WITH_PREFIX).sort(["tier", "weight_bin"])

In [83]:
with pl.Config(tbl_rows=100, tbl_width_chars=500):
    print(weight_default_high)

shape: (21, 6)
┌───────────────┬──────────┬────────────────┬──────┬────────┬───────┐
│ tier          ┆ tool     ┆ weight_bin     ┆ n    ┆ n_tier ┆ pct   │
│ ---           ┆ ---      ┆ ---            ┆ ---  ┆ ---    ┆ ---   │
│ str           ┆ str      ┆ str            ┆ i64  ┆ i64    ┆ f64   │
╞═══════════════╪══════════╪════════════════╪══════╪════════╪═══════╡
│ CheckAMG_high ┆ CheckAMG ┆ no_annotation  ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ no_reference   ┆ 9    ┆ 3218   ┆ 0.28  │
│ CheckAMG_high ┆ CheckAMG ┆ not_called     ┆ 33   ┆ 3218   ┆ 1.03  │
│ CheckAMG_high ┆ CheckAMG ┆ weight_0.4_0.6 ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ weight_0.6_0.8 ┆ 2926 ┆ 3218   ┆ 90.93 │
│ CheckAMG_high ┆ CheckAMG ┆ weight_ge_0.8  ┆ 250  ┆ 3218   ┆ 7.77  │
│ CheckAMG_high ┆ CheckAMG ┆ weight_lt_0.4  ┆ 0    ┆ 3218   ┆ 0.0   │
│ DRAMV_default ┆ DRAMV    ┆ no_annotation  ┆ 0    ┆ 664    ┆ 0.0   │
│ DRAMV_default ┆ DRAMV    ┆ no_reference   ┆ 91   ┆ 664    ┆ 13.7  │
│ DRA

In [84]:
category_default_high = category_composition(members_weights, DEFAULT_HIGH_WITH_PREFIX).sort(["tier", "pct"], descending=[False, True])

In [85]:
with pl.Config(tbl_rows=100, tbl_width_chars=500):
    print(category_default_high)

shape: (24, 6)
┌───────────────┬──────────┬────────────────┬──────┬────────┬───────┐
│ tier          ┆ tool     ┆ category       ┆ n    ┆ n_tier ┆ pct   │
│ ---           ┆ ---      ┆ ---            ┆ ---  ┆ ---    ┆ ---   │
│ str           ┆ str      ┆ str            ┆ i64  ┆ i64    ┆ f64   │
╞═══════════════╪══════════╪════════════════╪══════╪════════╪═══════╡
│ CheckAMG_high ┆ CheckAMG ┆ none_likely    ┆ 1601 ┆ 3218   ┆ 49.75 │
│ CheckAMG_high ┆ CheckAMG ┆ nucleotide     ┆ 1050 ┆ 3218   ┆ 32.63 │
│ CheckAMG_high ┆ CheckAMG ┆ methyl         ┆ 347  ┆ 3218   ┆ 10.78 │
│ CheckAMG_high ┆ CheckAMG ┆ glucan         ┆ 127  ┆ 3218   ┆ 3.95  │
│ CheckAMG_high ┆ CheckAMG ┆ lipid          ┆ 52   ┆ 3218   ┆ 1.62  │
│ CheckAMG_high ┆ CheckAMG ┆ none_no_weight ┆ 41   ┆ 3218   ┆ 1.27  │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ none_unlikely  ┆ 0    ┆ 3218   ┆ 0.0   │
│ DRAMV_default ┆ DRAMV    ┆ none_likely    ┆ 195  ┆ 664    ┆ 29.37 │
│ DRA

In [86]:
category_weight_default_high = (
    category_weight_composition(members_weights, DEFAULT_HIGH_WITH_PREFIX)
    .sort(["tier", "category", "weight_bin"])
)

In [87]:
with pl.Config(tbl_rows=200, tbl_width_chars=500):
    print(category_weight_default_high)

shape: (168, 7)
┌───────────────┬──────────┬────────────────┬────────────────┬──────┬────────┬───────┐
│ tier          ┆ tool     ┆ category       ┆ weight_bin     ┆ n    ┆ n_tier ┆ pct   │
│ ---           ┆ ---      ┆ ---            ┆ ---            ┆ ---  ┆ ---    ┆ ---   │
│ str           ┆ str      ┆ str            ┆ str            ┆ u64  ┆ i32    ┆ f64   │
╞═══════════════╪══════════╪════════════════╪════════════════╪══════╪════════╪═══════╡
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ no_annotation  ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ no_reference   ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ not_called     ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ weight_0.4_0.6 ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ weight_0.6_0.8 ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG ┆ essential      ┆ weight_ge_0.8  ┆ 0    ┆ 3218   ┆ 0.0   │
│ CheckAMG_high ┆ CheckAMG 

In [88]:
source_db_default_high = source_db_composition(
    amg_predictions_categorized,
    DEFAULT_HIGH_WITH_PREFIX
).sort(["tier", "database"])

In [89]:
with pl.Config(tbl_rows=200, tbl_width_chars=500):
    print(source_db_default_high)

shape: (18, 6)
┌───────────────┬──────────┬───────────┬──────┬────────┬───────┐
│ tier          ┆ tool     ┆ database  ┆ n    ┆ n_tier ┆ pct   │
│ ---           ┆ ---      ┆ ---       ┆ ---  ┆ ---    ┆ ---   │
│ str           ┆ str      ┆ str       ┆ i64  ┆ i64    ┆ f64   │
╞═══════════════╪══════════╪═══════════╪══════╪════════╪═══════╡
│ CheckAMG_high ┆ CheckAMG ┆ CAMPER    ┆ 32   ┆ 5520   ┆ 0.58  │
│ CheckAMG_high ┆ CheckAMG ┆ FOAM      ┆ 815  ┆ 5520   ┆ 14.76 │
│ CheckAMG_high ┆ CheckAMG ┆ KEGG      ┆ 2581 ┆ 5520   ┆ 46.76 │
│ CheckAMG_high ┆ CheckAMG ┆ METABOLIC ┆ 9    ┆ 5520   ┆ 0.16  │
│ CheckAMG_high ┆ CheckAMG ┆ Pfam      ┆ 2083 ┆ 5520   ┆ 37.74 │
│ CheckAMG_high ┆ CheckAMG ┆ dbCAN     ┆ 0    ┆ 5520   ┆ 0.0   │
│ DRAMV_default ┆ DRAM-V   ┆ CAMPER    ┆ 0    ┆ 2397   ┆ 0.0   │
│ DRAMV_default ┆ DRAM-V   ┆ FOAM      ┆ 0    ┆ 2397   ┆ 0.0   │
│ DRAMV_default ┆ DRAM-V   ┆ KEGG      ┆ 4    ┆ 2397   ┆ 0.17  │
│ DRAMV_default ┆ DRAM-V   ┆ METABOLIC ┆ 0    ┆ 2397   ┆ 0.0   │
│ DRAMV_de

### Members at any tier

Genes called at any tier by any tool.

In [90]:
all_tier_cols = list(ALL_TIERS)
any_tier = pl.any_horizontal([pl.col(c) for c in all_tier_cols])
members_all = (
    amg_predictions.filter(any_tier)
        .select(
            "gene", "scaffold", "source", "sample", "ecosystem",
            "genomad_viral", "in_strict_viral_region",
            "viral_gene_left_dist", "viral_gene_right_dist",
            "MGE_gene_left_dist", "MGE_gene_right_dist",
            "MGE_gene_left_V_score", "MGE_gene_right_V_score",
            *all_tier_cols,
            "DRAMV_auxiliary_score",
            "DRAMV_M", "DRAMV_T", "DRAMV_V", "DRAMV_A", "DRAMV_P", "DRAMV_B", "DRAMV_F",
        )
)

### Annotation database per gene across all tiers

For every gene called AMG at any tier, pool the `final_annot` hits from each tool that called it (at any tier), restricted to databases in CheckAMG's AMG reference. A gene is labeled with that database if only one is present, "Multiple DBs" if more than one, and "None" if none. Counts are aggregated per tier-membership pattern for the all-tier UpSet plot.

In [91]:
TOOL_TIER_COLS = {
    "CheckAMG": ["CheckAMG_high", "CheckAMG_medium", "CheckAMG_low"],
    "DRAM-V": ["DRAMV_default", "DRAMV_allow_T", "DRAMV_aux_4", "DRAMV_aux_4_allow_T"],
    "VIBRANT": ["VIBRANT_high", "VIBRANT_medium", "VIBRANT_low"],
}
GENE_KEYS = ["gene", "source", "sample", "ecosystem"]

calling_tools = (
    members_all.lazy()
    .select(*GENE_KEYS, *[pl.any_horizontal(cols).alias(tool) for tool, cols in TOOL_TIER_COLS.items()])
    .unpivot(index=GENE_KEYS, variable_name="tool", value_name="called")
    .filter(pl.col("called"))
    .drop("called")
)

gene_db_source = (
    calling_tools
    .join(
        annotations.lazy()
        .filter(pl.col("final_annot") & pl.col("database").is_in(checkamg_dbs))
        .select(*GENE_KEYS, "tool", "database")
        .unique(),
        on=[*GENE_KEYS, "tool"],
        how="left",
    )
    .group_by(GENE_KEYS)
    .agg(pl.col("database").drop_nulls().unique().alias("dbs"))
    .with_columns(
        pl.when(pl.col("dbs").list.len() == 0).then(pl.lit("None"))
          .when(pl.col("dbs").list.len() > 1).then(pl.lit("Multiple DBs"))
          .otherwise(pl.col("dbs").list.first())
          .alias("db_source")
    )
    .drop("dbs")
)

upset_db_all_tiers = (
    members_all.lazy()
    .select(*GENE_KEYS, *all_tier_cols)
    .join(gene_db_source, on=GENE_KEYS, how="left")
    .group_by([*all_tier_cols, "db_source"])
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)

assert upset_db_all_tiers["n"].sum() == members_all.height

In [92]:
with pl.Config(tbl_rows=30, tbl_width_chars=500):
    print(upset_db_all_tiers.group_by("db_source").agg(pl.col("n").sum()).sort("n", descending=True))

shape: (8, 2)
┌──────────────┬─────────┐
│ db_source    ┆ n       │
│ ---          ┆ ---     │
│ str          ┆ u64     │
╞══════════════╪═════════╡
│ Pfam         ┆ 1765972 │
│ Multiple DBs ┆ 200096  │
│ KEGG         ┆ 8995    │
│ FOAM         ┆ 1699    │
│ None         ┆ 983     │
│ METABOLIC    ┆ 80      │
│ CAMPER       ┆ 41      │
│ dbCAN        ┆ 4       │
└──────────────┴─────────┘


### Likely-AMG restricted overlap

A tool's default or high call is a likely AMG if its own best annotation carries `AMG_weight >= 0.6`. The three-way overlap is recomputed with the same tier columns, restricted to likely calls.

These counts differ from those in amg_benchmark_figures.Rmd, which gives 3,176 of 3,218 CheckAMG (high), 524 of 664 DRAM-V (default), and 398 of 1,270 VIBRANT (high) calls as likely AMGs. The two agree for CheckAMG and DRAM-V and give 450 against 398 for VIBRANT.

In [93]:
def passes_weight(prefix: str) -> pl.Expr:
    return pl.col(f"{prefix}_weight_bin").is_in(["weight_ge_0.8", "weight_0.6_0.8"])

members_likely = members_weights.with_columns([
    (pl.col("CheckAMG_high") & passes_weight("CheckAMG")).alias("CheckAMG_high_likely"),
    (pl.col("DRAMV_default") & passes_weight("DRAMV")  ).alias("DRAMV_default_likely"),
    (pl.col("VIBRANT_high")  & passes_weight("VIBRANT") ).alias("VIBRANT_high_likely"),
])

likely_venn_rows = []
for c in [False, True]:
    for d in [False, True]:
        for v in [False, True]:
            if not (c or d or v):
                continue
            expr = (
                (pl.col("CheckAMG_high_likely") == c)
                & (pl.col("DRAMV_default_likely") == d)
                & (pl.col("VIBRANT_high_likely") == v)
            )
            name = " & ".join([t for t, flag in zip(["CheckAMG", "DRAMV", "VIBRANT"], [c, d, v]) if flag])
            likely_venn_rows.append({"set": name, "CheckAMG": c, "DRAMV": d, "VIBRANT": v,
                                     "n": int(members_likely.filter(expr).height)})
members_likely_venn = pl.DataFrame(likely_venn_rows).sort("n", descending=True)

In [94]:
members_likely_venn

set,CheckAMG,DRAMV,VIBRANT,n
str,bool,bool,bool,i64
"""CheckAMG""",true,false,false,2997
"""DRAMV""",false,true,false,524
"""VIBRANT""",false,false,true,271
"""CheckAMG & VIBRANT""",true,false,true,179
"""DRAMV & VIBRANT""",false,true,true,0
"""CheckAMG & DRAMV""",true,true,false,0
"""CheckAMG & DRAMV & VIBRANT""",true,true,true,0


In [95]:
members_likely_totals = pl.DataFrame([
    {"tier": "CheckAMG_high",
     "n_all":        int(members_likely.filter(pl.col("CheckAMG_high")).height),
     "n_likely_amg": int(members_likely.filter(pl.col("CheckAMG_high") & pl.col("CheckAMG_high_likely")).height)},
    {"tier": "DRAMV_default",
     "n_all":        int(members_likely.filter(pl.col("DRAMV_default")).height),
     "n_likely_amg": int(members_likely.filter(pl.col("DRAMV_default") & pl.col("DRAMV_default_likely")).height)},
    {"tier": "VIBRANT_high",
     "n_all":        int(members_likely.filter(pl.col("VIBRANT_high")).height),
     "n_likely_amg": int(members_likely.filter(pl.col("VIBRANT_high") & pl.col("VIBRANT_high_likely")).height)},
]).with_columns(
    (pl.col("n_likely_amg") / pl.col("n_all") * 100).round(2).alias("pct_likely_amg")
)

In [96]:
members_likely_totals

tier,n_all,n_likely_amg,pct_likely_amg
str,i64,i64,f64
"""CheckAMG_high""",3218,3176,98.69
"""DRAMV_default""",664,524,78.92
"""VIBRANT_high""",1270,450,35.43


## Database sources of AMG predictions from each tool

In [97]:
checkamg_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("checkamg_results_raw.parquet"))
dramv_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("dramv_results_raw.parquet"))
vibrant_results_raw = pl.read_parquet(AMG_TABLES_DIR.joinpath("vibrant_results_raw.parquet"))

In [98]:
amg_database_kegg = set(amg_database.filter(pl.col("KO").is_not_null())["KO"].to_list())
amg_database_pfam = set(
    amg_database.filter(pl.col("PFAM").is_not_null())["PFAM"]
    .str.strip_chars()  # handles leading spaces seen in data
    .to_list()
)

amgs_filtered = amgs.filter(pl.col("metabolic_ratio") >= 0.6)
amgs_kegg = set(amgs_filtered.filter(pl.col("db") == "KEGG")["id"].to_list())
amgs_pfam = set(amgs_filtered.filter(pl.col("db") == "Pfam")["id"].to_list())

vibrant_ko_set = set(vibrant_amgs["ko"].to_list())

In [99]:
DBS = ["KEGG", "Pfam"]

In [100]:
def get_genes_with_db(tool: str, tier_col: str, db: str, amg_predictions: pl.DataFrame) -> pl.Series:
    tier_genes = (
        amg_predictions
        .filter(pl.col(tier_col))
        ["gene"]
        .to_list()
    )

    if tool == "DRAMV":
        prefix = "K" if db == "KEGG" else "PF"
        matched = (
            dramv_results_raw
            .filter(pl.col("gene").is_in(tier_genes))
            .filter(pl.col("gene_id").str.starts_with(prefix))
            ["gene"]
            .unique()
        )
    elif tool == "CheckAMG":
        col = "KEGG KO" if db == "KEGG" else "Pfam Accession"
        matched = (
            checkamg_results_raw
            .filter(pl.col("gene").is_in(tier_genes))
            .filter(pl.col(col).is_not_null())
            ["gene"]
            .unique()
        )
    elif tool == "VIBRANT":
        col = "AMG KO" if db == "KEGG" else "Pfam"
        matched = (
            vibrant_results_raw
            .filter(pl.col("gene").is_in(tier_genes))
            .filter(pl.col(col).is_not_null())
            ["gene"]
            .unique()
        )

    return matched

def get_genes_with_db_validated(tool: str, tier_col: str, db: str, amg_predictions: pl.DataFrame) -> pl.Series:
    tier_genes = (
        amg_predictions
        .filter(pl.col(tier_col))
        ["gene"]
        .to_list()
    )

    if tool == "DRAMV":
        prefix = "K" if db == "KEGG" else "PF"
        valid_ids = amg_database_kegg if db == "KEGG" else amg_database_pfam
        matched = (
            dramv_results_raw
            .filter(pl.col("gene").is_in(tier_genes))
            .filter(pl.col("gene_id").str.starts_with(prefix))
            .filter(pl.col("gene_id").is_in(valid_ids))
            ["gene"]
            .unique()
        )
    elif tool == "CheckAMG":
        col = "KEGG KO" if db == "KEGG" else "Pfam Accession"
        valid_ids = amgs_kegg if db == "KEGG" else amgs_pfam
        matched = (
            checkamg_results_raw
            .filter(pl.col("gene").is_in(tier_genes))
            .filter(pl.col(col).is_not_null())
            .filter(pl.col(col).is_in(valid_ids))
            ["gene"]
            .unique()
        )
    elif tool == "VIBRANT":
        if db == "KEGG":
            matched = (
                vibrant_results_raw
                .filter(pl.col("gene").is_in(tier_genes))
                .filter(pl.col("AMG KO").is_not_null())
                .filter(pl.col("AMG KO").is_in(vibrant_ko_set))
                ["gene"]
                .unique()
            )
        else:  # Pfam has no VIBRANT validation list
            matched = pl.Series("gene", [], dtype=pl.String)

    return matched

In [101]:
for tier_col, tool in SOURCE_TIERS:
    for db in DBS:
        tier_total = amg_predictions.filter(pl.col(tier_col))["gene"].n_unique()

        primary_genes = get_genes_with_db(tool, tier_col, db, amg_predictions)
        primary_in_ref = get_genes_with_db_validated(tool, tier_col, db, amg_predictions)

        print(f"{tool} [{tier_col}] | {db}: "
              f"{len(primary_genes)} / {tier_total} unique genes "
              f"({len(primary_genes)/tier_total*100:.2f}%)")
        print(f"  - {tool} AMG ref: "
              f"{len(primary_in_ref)} / {tier_total} unique genes "
              f"({len(primary_in_ref)/tier_total*100:.2f}%)")

        for other_col in OTHER_TIER_FLAGS[tool]:
            other_col_name = other_col.meta.output_name()
            other_total = amg_predictions.filter(other_col)["gene"].n_unique()

            other_genes = get_genes_with_db(tool, other_col_name, db, amg_predictions)
            other_in_ref = get_genes_with_db_validated(tool, other_col_name, db, amg_predictions)

            print(f"  {tool} [{other_col_name}] | {db}: "
                  f"{len(other_genes)} / {other_total} genes "
                  f"({len(other_genes)/other_total*100:.2f}%)")
            print(f"    - in {tool} AMG ref: "
                  f"{len(other_in_ref)} / {other_total} genes "
                  f"({len(other_in_ref)/other_total*100:.2f}%)")

CheckAMG [CheckAMG_high] | KEGG: 2611 / 3217 unique genes (81.16%)
  - CheckAMG AMG ref: 2085 / 3217 unique genes (64.81%)
  CheckAMG [CheckAMG_medium] | KEGG: 5426 / 6699 genes (81.00%)
    - in CheckAMG AMG ref: 4151 / 6699 genes (61.96%)


  CheckAMG [CheckAMG_low] | KEGG: 151497 / 167995 genes (90.18%)
    - in CheckAMG AMG ref: 122800 / 167995 genes (73.10%)
CheckAMG [CheckAMG_high] | Pfam: 2197 / 3217 unique genes (68.29%)
  - CheckAMG AMG ref: 957 / 3217 unique genes (29.75%)
  CheckAMG [CheckAMG_medium] | Pfam: 4697 / 6699 genes (70.11%)
    - in CheckAMG AMG ref: 2134 / 6699 genes (31.86%)


  CheckAMG [CheckAMG_low] | Pfam: 135573 / 167995 genes (80.70%)
    - in CheckAMG AMG ref: 59576 / 167995 genes (35.46%)
DRAMV [DRAMV_default] | KEGG: 10 / 664 unique genes (1.51%)
  - DRAMV AMG ref: 1 / 664 unique genes (0.15%)
  DRAMV [DRAMV_allow_T] | KEGG: 220 / 10729 genes (2.05%)
    - in DRAMV AMG ref: 20 / 10729 genes (0.19%)


  DRAMV [DRAMV_aux_4] | KEGG: 9533 / 114024 genes (8.36%)
    - in DRAMV AMG ref: 397 / 114024 genes (0.35%)


  DRAMV [DRAMV_aux_4_allow_T] | KEGG: 98774 / 1590821 genes (6.21%)
    - in DRAMV AMG ref: 4000 / 1590821 genes (0.25%)
DRAMV [DRAMV_default] | Pfam: 664 / 664 unique genes (100.00%)
  - DRAMV AMG ref: 663 / 664 unique genes (99.85%)
  DRAMV [DRAMV_allow_T] | Pfam: 10725 / 10729 genes (99.96%)
    - in DRAMV AMG ref: 10674 / 10729 genes (99.49%)


  DRAMV [DRAMV_aux_4] | Pfam: 113916 / 114024 genes (99.91%)
    - in DRAMV AMG ref: 113602 / 114024 genes (99.63%)


  DRAMV [DRAMV_aux_4_allow_T] | Pfam: 1589349 / 1590821 genes (99.91%)
    - in DRAMV AMG ref: 1583876 / 1590821 genes (99.56%)
VIBRANT [VIBRANT_high] | KEGG: 1270 / 1270 unique genes (100.00%)
  - VIBRANT AMG ref: 1270 / 1270 unique genes (100.00%)
  VIBRANT [VIBRANT_medium] | KEGG: 3715 / 3715 genes (100.00%)
    - in VIBRANT AMG ref: 3715 / 3715 genes (100.00%)
  VIBRANT [VIBRANT_low] | KEGG: 12623 / 12623 genes (100.00%)
    - in VIBRANT AMG ref: 12623 / 12623 genes (100.00%)
VIBRANT [VIBRANT_high] | Pfam: 1079 / 1270 unique genes (84.96%)
  - VIBRANT AMG ref: 0 / 1270 unique genes (0.00%)
  VIBRANT [VIBRANT_medium] | Pfam: 3078 / 3715 genes (82.85%)
    - in VIBRANT AMG ref: 0 / 3715 genes (0.00%)
  VIBRANT [VIBRANT_low] | Pfam: 10235 / 12623 genes (81.08%)
    - in VIBRANT AMG ref: 0 / 12623 genes (0.00%)


## Additional numbers

Assembled from the tables above for easy reporting.

The VIBRANT (high) likely-AMG count printed below (450) follows the definition in the likely-AMG section above and differs from the 398 in amg_benchmark_figures.Rmd (see the likely-AMG section).

In [102]:
def _first(df, col_name, col_val, target):
    return df.filter(pl.col(col_name) == col_val)[target][0]

stats = {
    "CheckAMG (high) total calls":             int(_first(counts_tier, "tier", "CheckAMG_high", "n_amg")),
    "CheckAMG (high) weight >= 0.6 (likely)":   int(_first(members_likely_totals, "tier", "CheckAMG_high", "n_likely_amg")),
    "CheckAMG (high) percent likely":          float(_first(members_likely_totals, "tier", "CheckAMG_high", "pct_likely_amg")),
    "DRAM-V (default) total calls":            int(_first(counts_tier, "tier", "DRAMV_default", "n_amg")),
    "DRAM-V (default) weight >= 0.6 (likely)":  int(_first(members_likely_totals, "tier", "DRAMV_default", "n_likely_amg")),
    "DRAM-V (default) percent likely":         float(_first(members_likely_totals, "tier", "DRAMV_default", "pct_likely_amg")),
    "VIBRANT (high) total calls":              int(_first(counts_tier, "tier", "VIBRANT_high", "n_amg")),
    "VIBRANT (high) weight >= 0.6 (likely)":    int(_first(members_likely_totals, "tier", "VIBRANT_high", "n_likely_amg")),
    "VIBRANT (high) percent likely":           float(_first(members_likely_totals, "tier", "VIBRANT_high", "pct_likely_amg")),
}
stats["CheckAMG / VIBRANT ratio (likely counts)"] = round(
    stats["CheckAMG (high) weight >= 0.6 (likely)"] / max(stats["VIBRANT (high) weight >= 0.6 (likely)"], 1), 2)
stats["CheckAMG / DRAM-V ratio (likely counts)"]  = round(
    stats["CheckAMG (high) weight >= 0.6 (likely)"] / max(stats["DRAM-V (default) weight >= 0.6 (likely)"], 1), 2)

In [103]:
for k, v in stats.items():
    print(f"{k:<45} {v}")

CheckAMG (high) total calls                   3218
CheckAMG (high) weight >= 0.6 (likely)        3176
CheckAMG (high) percent likely                98.69
DRAM-V (default) total calls                  664
DRAM-V (default) weight >= 0.6 (likely)       524
DRAM-V (default) percent likely               78.92
VIBRANT (high) total calls                    1270
VIBRANT (high) weight >= 0.6 (likely)         450
VIBRANT (high) percent likely                 35.43
CheckAMG / VIBRANT ratio (likely counts)      7.06
CheckAMG / DRAM-V ratio (likely counts)       6.06


## All-tier gene-level exports

`per_tool_gene` covers only default and high tier members. The cells below rebuild the per-gene tables over every gene called at any tier and write them for amg_benchmark_figures.Rmd, which computes all percentages. The annotation table is filtered to final annotations before the join, which keeps the intermediate at about ten million rows.

In [104]:
GENE_KEY = ["gene", "source", "sample", "ecosystem"]

assert members_all.select(GENE_KEY).unique().height == members_all.height, "gene key is not unique in members_all"

members_all.write_parquet(OUTPUT_TABLES_DIR.joinpath("AMG_predictions_gene_level.parquet"))
print(f"AMG_predictions_gene_level.parquet: {members_all.height:,} rows, {members_all.width} columns")
print(f"  default/high member gene keys for comparison: {members.select(GENE_KEY).unique().height:,}")


AMG_predictions_gene_level.parquet: 1,977,870 rows, 31 columns
  default/high member gene keys for comparison: 4,931


In [105]:
final_annotations_all_tiers = (
    annotations.filter(pl.col("final_annot"))
    .join(members_all.select(GENE_KEY), on=GENE_KEY, how="inner")
    .with_columns(
        pl.when(pl.col("database") == "Pfam")
          .then(pl.col("hit_id").str.split(".").list.first())
          .otherwise(pl.col("hit_id"))
          .alias("id_key"),
    )
    .join(amgs_slim, left_on=["database", "id_key"], right_on=["db", "id_key"], how="left")
    .join(filters_slim, left_on=["database", "id_key"], right_on=["db", "id_key"], how="left")
    .with_columns([pl.col(c).fill_null(False) for c in FILTER_CATEGORIES])
    .select(["gene", "sample", "source", "ecosystem", "tool", "database", "hit_id", "AMG_weight"]
            + FILTER_CATEGORIES)
)

print(f"final annotations at all tiers: {final_annotations_all_tiers.height:,} rows, "
      f"{final_annotations_all_tiers.width} columns")
print(f"  columns: {final_annotations_all_tiers.columns}")
print(f"  for comparison, the default/high basis: {final_annots.height:,} rows")
print(final_annotations_all_tiers.group_by("tool").agg(pl.len().alias("n_rows")).sort("tool"))
print(final_annotations_all_tiers.group_by("database").agg(pl.len().alias("n_rows"))
      .sort("n_rows", descending=True))

final annotations at all tiers: 9,491,928 rows, 13 columns
  columns: ['gene', 'sample', 'source', 'ecosystem', 'tool', 'database', 'hit_id', 'AMG_weight', 'filter_essential', 'filter_glucan', 'filter_lipid', 'filter_methyl', 'filter_nucleotide']
  for comparison, the default/high basis: 37,247 rows
shape: (3, 2)
┌──────────┬─────────┐
│ tool     ┆ n_rows  │
│ ---      ┆ ---     │
│ str      ┆ u64     │
╞══════════╪═════════╡
│ CheckAMG ┆ 1752214 │
│ DRAM-V   ┆ 7717042 │
│ VIBRANT  ┆ 22672   │
└──────────┴─────────┘
shape: (7, 2)
┌───────────┬─────────┐
│ database  ┆ n_rows  │
│ ---       ┆ ---     │
│ str       ┆ u64     │
╞═══════════╪═════════╡
│ Pfam      ┆ 8190802 │
│ KEGG      ┆ 586763  │
│ PHROG     ┆ 434198  │
│ FOAM      ┆ 222874  │
│ CAMPER    ┆ 46251   │
│ METABOLIC ┆ 9904    │
│ dbCAN     ┆ 1136    │
└───────────┴─────────┘


In [106]:
nonoverlap_per_gene = classify_nonoverlap(members_annot, return_per_gene=True)
nonoverlap_per_gene.write_parquet(OUTPUT_TABLES_DIR.joinpath("AMG_nonoverlap_per_gene.parquet"))
print(f"AMG_nonoverlap_per_gene.parquet: {nonoverlap_per_gene.height:,} rows, "
      f"{nonoverlap_per_gene['source_tier'].n_unique()} source tiers, "
      f"{nonoverlap_per_gene['other_tool'].n_unique()} other tools")
print(nonoverlap_per_gene.head(4))


AMG_nonoverlap_per_gene.parquet: 10,304 rows, 3 source tiers, 3 other tools
shape: (4, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ gene       ┆ source     ┆ sample     ┆ ecosystem ┆ source_ti ┆ other_too ┆ category  ┆ likely_fa │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ er        ┆ l         ┆ ---       ┆ lse_posit │
│ str        ┆ str        ┆ str        ┆ str       ┆ ---       ┆ ---       ┆ str       ┆ ive       │
│            ┆            ┆            ┆           ┆ str       ┆ str       ┆           ┆ ---       │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ bool      │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Ga0485157_ ┆ Mixed meta ┆ freshwater ┆ Aquatic   ┆ CheckAMG_ ┆ DRAMV     ┆ d. same   ┆ false     │
│ 0000020_8  ┆ genomes    ┆ _Ga0485157 ┆           ┆ high      ┆           ┆ db        ┆           │
│

### Curated bitscore thresholds for the all-tier annotation export

Each annotation row is judged against the curated threshold of the database the tool searched, so that a call can be scored on whether its supporting annotation clears the threshold its own reference publishes. KEGG rows use the KOfam `ko_list` threshold. Pfam rows use the gathering cutoff embedded in the profile, taken from the Pfam build the tool actually searched, which for VIBRANT is Pfam-A v32. FOAM, METABOLIC and CAMPER use the cutoff tables `hmm_annotate.py` loads, full cutoff preferred and domain cutoff as fallback. DRAM-V Pfam rows carry the hmmscan `--cut_ga` verdict from the re-run rather than a comparison against DRAM-V's own MMseqs2 score, and `threshold_basis` records that. PHROG and dbCAN publish no curated bitscore threshold, so all three threshold columns are null for those rows.

In [107]:
import csv as _csv

_DB = OUTPUT_TABLES_DIR.parent.parent / "CheckAMG_annotate_db_v1.1_20260316"
_KO_V91 = "/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks/metabolism_benchmark/refs/kofam_v91_2019-08-10_ko_list.tsv"
_CK_KEGG = _DB / "KEGG_cutoffs.tsv"
_DRAM_KO = "/storage2/databases/dram-latest/v1.5.0/kofam_ko_list.tsv"
_VIB_PFAM = "/storage2/databases/VIBRANT/databases/Pfam-A_v32.HMM"
_PF_CUR = _DB / "Pfam-A.hmm"
_KEY = ["gene", "source", "sample", "ecosystem"]


def _read_kofam(path):
    ids, thr, styp = [], [], []
    with open(path, encoding="utf-8", errors="replace") as fh:
        for row in _csv.DictReader(fh, delimiter="\t"):
            t = (row.get("threshold") or "").strip()
            if t and t != "-":
                ids.append(row["knum"].strip())
                thr.append(float(t))
                styp.append((row.get("score_type") or "").strip())
    return pl.DataFrame({"hit_id": ids, "thr": thr, "score_type": styp})


def _read_ga(path):
    accs, gas = [], []
    acc = None
    with open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("ACC "):
                acc = line.split(None, 1)[1].strip()
            elif line.startswith("GA "):
                accs.append(acc)
                gas.append(float(line.split()[1]))
    return (pl.DataFrame({"acc_base": accs, "thr": gas})
              .with_columns(pl.col("acc_base").str.replace(r"\.\d+$", ""))
              .unique(subset=["acc_base"]))


def _effective(path):
    t = pl.read_csv(path, separator="\t", infer_schema_length=20000)
    return t.select(pl.col("id").alias("hit_id"),
                    pl.coalesce([pl.col("cutoff_full"), pl.col("cutoff_domain")]).alias("thr"))


_ex = final_annotations_all_tiers
_cols13 = _ex.columns
_n0 = _ex.height

_src = (pl.scan_parquet(AMG_TABLES_DIR.joinpath("amg_all_annotations.parquet"))
          .filter(pl.col("final_annot"))
          .group_by(_KEY + ["tool", "database", "hit_id"])
          .agg(pl.col("bitscore").max().alias("bitscore"),
               pl.col("evalue").min().alias("evalue"))
          .collect())
_ex = _ex.join(_src, on=_KEY + ["tool", "database", "hit_id"], how="left")
assert _ex.height == _n0, "bitscore join changed the row count"

# Each tool is judged against the KEGG thresholds shipped with the release it searched
_v91_full = _read_kofam(_KO_V91)
_dram_ko_full = _read_kofam(_DRAM_KO)
_ck_kegg = (pl.read_csv(_CK_KEGG, separator="\t", infer_schema_length=20000)
              .select(pl.col("id").alias("hit_id"), pl.col("threshold").alias("thr")))
# One full-sequence score per row, so score_type is recorded where a domain-type KO is judged on it
_v91 = _v91_full.drop("score_type")
_dram_ko = _dram_ko_full.drop("score_type")
_pf_cur = _read_ga(_PF_CUR)
_pf_v32 = _read_ga(_VIB_PFAM)
_foam = _effective(_DB / "FOAM_cutoffs.tsv")
_metab = _effective(_DB / "METABOLIC_cutoffs.tsv")
_camper = _effective(_DB / "CAMPER_cutoffs.tsv")

_ex = _ex.with_columns(pl.col("hit_id").str.replace(r"\.\d+$", "").alias("acc_base"))
_T, _D = pl.col("tool"), pl.col("database")
_RULES = [
    ((_T == "CheckAMG") & (_D == "KEGG"), _ck_kegg, "hit_id", "checkamg_kegg_cutoffs_v1.1"),
    ((_T == "CheckAMG") & (_D == "FOAM"), _foam, "hit_id", "foam"),
    ((_T == "CheckAMG") & (_D == "METABOLIC"), _metab, "hit_id", "metabolic"),
    ((_T == "CheckAMG") & (_D == "CAMPER"), _camper, "hit_id", "camper"),
    ((_T == "CheckAMG") & (_D == "Pfam"), _pf_cur, "acc_base", "pfam_ga"),
    ((_T == "DRAM-V") & (_D == "KEGG"), _dram_ko, "hit_id", "kofam"),
    ((_T == "DRAM-V") & (_D == "Pfam"), _pf_cur, "acc_base", "dramv_hmmscan_cut_ga"),
    ((_T == "VIBRANT") & (_D == "KEGG"), _v91, "hit_id", "kofam_v91"),
    ((_T == "VIBRANT") & (_D == "Pfam"), _pf_v32, "acc_base", "pfam_ga"),
]
_parts, _cov = [], None
for _mask, _tbl, _on, _basis in _RULES:
    _p = _ex.filter(_mask).join(_tbl, on=_on, how="left")
    _parts.append(_p.with_columns(pl.when(pl.col("thr").is_not_null()).then(pl.lit(_basis))
                                    .otherwise(pl.lit(None, dtype=pl.Utf8)).alias("threshold_basis")))
    _cov = _mask if _cov is None else (_cov | _mask)
_rest = _ex.filter(~_cov).with_columns(pl.lit(None, dtype=pl.Float64).alias("thr"),
                                       pl.lit(None, dtype=pl.Utf8).alias("threshold_basis"))
_out = pl.concat(_parts + [_rest], how="vertical_relaxed")
assert _out.height == _n0, "threshold partition lost rows"

_out = _out.rename({"thr": "db_threshold"}).with_columns(
    pl.when(pl.col("db_threshold").is_null() | pl.col("bitscore").is_null())
      .then(pl.lit(None, dtype=pl.Boolean))
      .otherwise(pl.col("bitscore") >= pl.col("db_threshold")).alias("passes_db_threshold"))

_dv = (pl.read_parquet(OUTPUT_TABLES_DIR.parent.joinpath("DRAMV_pfam", "dramv_pfam_amg_calls.parquet"),
                       columns=["new_gene", "sample", "pfam_acc", "support_class"])
         .with_columns((pl.col("support_class") == "hmmer_supported").alias("dv_pass"))
         .group_by(["new_gene", "sample", "pfam_acc"]).agg(pl.col("dv_pass").max().alias("dv_pass"))
         .rename({"new_gene": "gene", "pfam_acc": "acc_base"}))
_out = _out.join(_dv, on=["gene", "sample", "acc_base"], how="left")
assert _out.height == _n0, "DRAM-V Pfam join changed the row count"
_dvmask = (pl.col("tool") == "DRAM-V") & (pl.col("database") == "Pfam")
_out = _out.with_columns(
    pl.when(_dvmask).then(pl.col("dv_pass").fill_null(False))
      .otherwise(pl.col("passes_db_threshold")).alias("passes_db_threshold")).drop(["dv_pass", "acc_base"])

_final = _cols13 + ["bitscore", "evalue", "db_threshold", "passes_db_threshold", "threshold_basis"]
_out = _out.select(_final)
assert _out.columns == _final and _out.height == _n0
# Written in parts so each file stays under GitHub's 100 MB limit
EXPORT_PARTS = 3
rows_per_part = -(-_out.height // EXPORT_PARTS)
for part_number, part in enumerate(_out.iter_slices(n_rows=rows_per_part), start=1):
    part.write_parquet(OUTPUT_TABLES_DIR.joinpath(f"AMG_final_annotations_all_tiers_part{part_number}.parquet"))
print(f"AMG_final_annotations_all_tiers_part1-{EXPORT_PARTS}.parquet: {_out.height:,} rows, {_out.width} columns")

_tab = (_out.group_by(["tool", "database"])
            .agg(pl.len().alias("rows"),
                 pl.col("passes_db_threshold").sum().alias("true"),
                 (pl.col("passes_db_threshold") == False).sum().alias("false"),
                 pl.col("passes_db_threshold").null_count().alias("null"),
                 pl.col("threshold_basis").drop_nulls().first().alias("basis"))
            .sort(["tool", "database"]))
print(_tab)
assert _out.filter((pl.col("tool") == "CheckAMG") & (pl.col("database") == "Pfam")
                   & (pl.col("passes_db_threshold") == False)).height == 0, \
    "CheckAMG enforces Pfam GA, so no row should fail it"
assert _out.filter(pl.col("database").is_in(["PHROG", "dbCAN"])
                   & pl.col("db_threshold").is_not_null()).height == 0, \
    "PHROG and dbCAN publish no curated threshold"
_styp = pl.concat([_v91_full.select("hit_id", "score_type"),
                   _dram_ko_full.select("hit_id", "score_type")]).unique(subset=["hit_id"])
_dom = (_out.filter((pl.col("database") == "KEGG") & pl.col("db_threshold").is_not_null())
            .join(_styp, on="hit_id", how="left")
            .group_by(["tool", "score_type"]).agg(pl.len().alias("rows")).sort(["tool", "score_type"]))
print("KEGG rows by KOfam score_type, domain-type rows judged on the full-sequence score:")
print(_dom)
print("threshold column guards passed")

AMG_final_annotations_all_tiers_part1-3.parquet: 9,491,928 rows, 18 columns


shape: (11, 7)
┌──────────┬───────────┬─────────┬────────┬─────────┬────────┬────────────────────────────┐
│ tool     ┆ database  ┆ rows    ┆ true   ┆ false   ┆ null   ┆ basis                      │
│ ---      ┆ ---       ┆ ---     ┆ ---    ┆ ---     ┆ ---    ┆ ---                        │
│ str      ┆ str       ┆ u64     ┆ u64    ┆ u64     ┆ u64    ┆ str                        │
╞══════════╪═══════════╪═════════╪════════╪═════════╪════════╪════════════════════════════╡
│ CheckAMG ┆ CAMPER    ┆ 46251   ┆ 8865   ┆ 37386   ┆ 0      ┆ camper                     │
│ CheckAMG ┆ FOAM      ┆ 222874  ┆ 162490 ┆ 60384   ┆ 0      ┆ foam                       │
│ CheckAMG ┆ KEGG      ┆ 524859  ┆ 362287 ┆ 158259  ┆ 4313   ┆ checkamg_kegg_cutoffs_v1.1 │
│ CheckAMG ┆ METABOLIC ┆ 9904    ┆ 5915   ┆ 3396    ┆ 593    ┆ metabolic                  │
│ CheckAMG ┆ PHROG     ┆ 434198  ┆ 0      ┆ 0       ┆ 434198 ┆ null                       │
│ …        ┆ …         ┆ …       ┆ …      ┆ …       ┆ …      ┆ … 